# v11 repaired - SFT parent and gate

HaNoRec (pinned `587face7`) CF-hardness reranker over frozen LLM2Rec + SASRec top-20 on Amazon Games.
This notebook is generated by `scripts/build_hanorec_notebook.py`; sources below are embedded and hash-checked.

**Repairs vs the exploratory run:** SFT now learns the binary Yes/No task (50% Yes / 50% No, as upstream);
DPO policy runs with dropout off (LLaMA-Factory default); LoRA is language-model only (vision tower frozen);
equal scores keep the retriever order. **Adaptations are recorded, not hidden:** Kaggle serves a T4 (bf16 emulated),
with fixed `max_pixels=16384`, four SFT epochs and one DPO epoch. New repairs: real-user exclusions,
clustered uncertainty, null-based permutation p-values, and clean/noisy learning probes.
**Not a confirmation:** one seed, validation only, test never scored.

## 1. Environment and GPU gate

In [ ]:
import hashlib, json, os, subprocess, sys, time
from pathlib import Path

NOTEBOOK_T0 = time.monotonic()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0", "pillow==11.0.0"], check=True)
import torch, transformers, peft

assert torch.cuda.is_available(), "GPU required"
GPU = {
    "name": torch.cuda.get_device_name(0),
    "total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    "capability": list(torch.cuda.get_device_capability(0)),
    "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
}
# torch.cuda.is_bf16_supported() is True on T4 by emulation, so it proves nothing. Kaggle's kernel API only
# offers T4/P100 (NvidiaL4 is silently served as a T4), so bf16 is emulated: recorded as an adaptation.
GPU["bf16_native"] = GPU["capability"][0] >= 8
print(GPU)
assert GPU["name"] and torch.cuda.is_bf16_supported(), "bf16 (even emulated) is required by the protocol"

## 2. Embedded, hash-verified sources

In [ ]:
EMBEDDED_SOURCES = {"prep.py": "...", "train.py": "...", "analysis.py": "IiIiVmFsaWRhdGlvbi1vbmx5IHBhaXJlZCBzdGF0aXN0aWNzIGZvciB0aGUgcmVwYWlyZWQgdjExIGdhdGUgKHN0YW5kYXJkIGxpYnJhcnkgb25seSkuCgpNZXRyaWNzIHVzZSB0aGUgZnVsbCBldmVudCBjb2hvcnQ6IHRhcmdldHMgYWJzZW50IGZyb20gdG9wLTIwIGNvbnRyaWJ1dGUgemVyby4KV2hlbiByZWFsLXVzZXIgY2x1c3RlciBJRHMgYXJlIHN1cHBsaWVkLCB1bmNlcnRhaW50eSByZXNhbXBsZXMgd2hvbGUgdXNlcnMuClRoZSBtZWFuIHJlbWFpbnMgZXZlbnQtd2VpZ2h0ZWQ7IHRyYWluaW5nLXNlZWQgdW5jZXJ0YWludHkgaXMgbm90IGVzdGltYXRlZCBoZXJlLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGl0ZXJ0b29scwppbXBvcnQgbWF0aAppbXBvcnQgcmFuZG9tCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIEl0ZXJhYmxlCgpNRVRSSUNTID0gKCJuZGNnQDEwIiwgInJlY2FsbEAxMCIpCk1JTl9NRUFOSU5HRlVMX05EQ0dfREVMVEEgPSAwLjAwMiAgIyBQaGFzZSAxIGZyb3plbiB0aHJlc2hvbGQgKHBsYW5zLzI2MDkyMC0qL3BoYXNlLTAxLXN0YXJ0Lm1kKQoKCmRlZiBfa2V5KHJvdzogZGljdFtzdHIsIEFueV0pIC0+IHR1cGxlW3N0ciwgaW50XToKICAgIHJldHVybiAoc3RyKHJvd1sidXNlcl9pZCJdKSwgaW50KHJvd1sidGFyZ2V0Il0pKQoKCmRlZiBmdWxsX2NvaG9ydF92YWx1ZXMoY29ob3J0OiBsaXN0W2RpY3Rbc3RyLCBBbnldXSwgcHJlZGljdGlvbnM6IEl0ZXJhYmxlW2RpY3Rbc3RyLCBBbnldXSkgLT4gbGlzdFtkaWN0W3N0ciwgQW55XV06CiAgICAiIiJFeHBhbmQgb25seSBub24taW5mb3JtYXRpdmUgbWlzc2luZyBldmVudHM7IHByZXNlcnZlIHJlYWwtdXNlciBjbHVzdGVycyBmb3IgcmVzYW1wbGluZy4iIiIKICAgIGJ5X2tleSA9IHt9CiAgICBmb3Igcm93IGluIHByZWRpY3Rpb25zOgogICAgICAgIGtleSA9IF9rZXkocm93KQogICAgICAgIGlmIGtleSBpbiBieV9rZXk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImR1cGxpY2F0ZSBwcmVkaWN0aW9uIGV2ZW50IikKICAgICAgICBieV9rZXlba2V5XSA9IHJvdwogICAgY29ob3J0X2tleXMgPSBbX2tleShyb3cpIGZvciByb3cgaW4gY29ob3J0XQogICAgaWYgbGVuKHNldChjb2hvcnRfa2V5cykpICE9IGxlbihjb2hvcnRfa2V5cyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZHVwbGljYXRlIGNvaG9ydCBldmVudCIpCiAgICBpZiBzZXQoYnlfa2V5KSAtIHNldChjb2hvcnRfa2V5cyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicHJlZGljdGlvbnMgb3V0c2lkZSB0aGUgdmFsaWRhdGlvbiBjb2hvcnQiKQogICAgdmFsdWVzID0gW10KICAgIGZvciByb3cgaW4gY29ob3J0OgogICAgICAgIHByZWRpY3Rpb24gPSBieV9rZXkuZ2V0KF9rZXkocm93KSkKICAgICAgICBpZiBwcmVkaWN0aW9uIGlzIE5vbmUgYW5kIGZsb2F0KHJvd1siY2FuZGlkYXRlX3JlY2FsbEAyMCJdKSAhPSAwLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImluZm9ybWF0aXZlIGV2ZW50IGhhcyBubyBwcmVkaWN0aW9uIikKICAgICAgICBpZiBwcmVkaWN0aW9uIGlzIG5vdCBOb25lIGFuZCAiY2x1c3Rlcl9pZCIgaW4gcm93IGFuZCBwcmVkaWN0aW9uLmdldCgiY2x1c3Rlcl9pZCIpICE9IHJvd1siY2x1c3Rlcl9pZCJdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcmVkaWN0aW9uIHVzZXIgY2x1c3RlciBkaWZmZXJzIGZyb20gY29ob3J0IikKICAgICAgICB2YWx1ZSA9IHttZXRyaWM6IGZsb2F0KHByZWRpY3Rpb25bbWV0cmljXSkgaWYgcHJlZGljdGlvbiBlbHNlIDAuMAogICAgICAgICAgICAgICAgIGZvciBtZXRyaWMgaW4gKCpNRVRSSUNTLCAiY2FuZGlkYXRlX3JlY2FsbEAyMCIpfQogICAgICAgIGlmIGFueShub3QgbWF0aC5pc2Zpbml0ZSh2KSBvciBub3QgMCA8PSB2IDw9IDEgZm9yIHYgaW4gdmFsdWUudmFsdWVzKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtZXRyaWMgbXVzdCBiZSBmaW5pdGUgYW5kIGJldHdlZW4gemVybyBhbmQgb25lIikKICAgICAgICBpZiAiY2x1c3Rlcl9pZCIgaW4gcm93OgogICAgICAgICAgICB2YWx1ZVsiY2x1c3Rlcl9pZCJdID0gcm93WyJjbHVzdGVyX2lkIl0KICAgICAgICB2YWx1ZXMuYXBwZW5kKHZhbHVlKQogICAgcmV0dXJuIHZhbHVlcwoKCmRlZiBzdW1tYXJpemUodmFsdWVzOiBsaXN0W2RpY3Rbc3RyLCBmbG9hdF1dKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgaWYgbm90IHZhbHVlczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlbXB0eSBjb2hvcnQiKQogICAgcmV0dXJuIHttZXRyaWM6IHN1bSh2W21ldHJpY10gZm9yIHYgaW4gdmFsdWVzKSAvIGxlbih2YWx1ZXMpIGZvciBtZXRyaWMgaW4gKCpNRVRSSUNTLCAiY2FuZGlkYXRlX3JlY2FsbEAyMCIpfQoKCmRlZiBwYWlyZWRfYm9vdHN0cmFwKAogICAgYTogbGlzdFtkaWN0W3N0ciwgQW55XV0sCiAgICBiOiBsaXN0W2RpY3Rbc3RyLCBBbnldXSwKICAgIG1ldHJpYzogc3RyID0gIm5kY2dAMTAiLAogICAgbjogaW50ID0gMjAwMDAsCiAgICBzZWVkOiBpbnQgPSAyMDI2MTAwMSwKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICIiIkV2ZW50LXdlaWdodGVkIG1lYW4gZGVsdGEgd2l0aCB1c2VyLWNsdXN0ZXIgYm9vdHN0cmFwIENJIGFuZCBhIG51bGwgc2lnbi1mbGlwIHRlc3QuCgogICAgU2lnbiBmbGlwcyBhY3Qgb24gd2hvbGUtdXNlciBwYWlyZWQgZGVsdGFzLiBUaGlzIHRlc3RzIHdpdGhpbi11c2VyIGxhYmVsIGV4Y2hhbmdlYWJpbGl0eSwKICAgIG5vdCB0cmFpbmluZy1zZWVkIHVuY2VydGFpbnR5LiBFeGFjdCBlbnVtZXJhdGlvbiBpcyB1c2VkIGZvciA8PTE2IG5vbnplcm8gY2x1c3RlcnMuCiAgICBVbmNsdXN0ZXJlZCBzeW50aGV0aWMvbGVnYWN5IGNhbGxlcnMgYXJlIGV4cGxpY2l0bHkgbGFiZWxsZWQgZXZlbnQtbGV2ZWwuCiAgICAiIiIKICAgIGlmIGxlbihhKSAhPSBsZW4oYikgb3Igbm90IGEgb3IgbiA8IDEwMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwYWlyZWQgY29ob3J0cyBtdXN0IGJlIGVxdWFsLWxlbmd0aCwgbm9uLWVtcHR5OyBuIG11c3QgYmUgPj0xMDAiKQogICAgY2x1c3RlcmVkID0gYW55KCJjbHVzdGVyX2lkIiBpbiByb3cgZm9yIHJvdyBpbiBhICsgYikKICAgIGlmIGNsdXN0ZXJlZCBhbmQgYW55KCJjbHVzdGVyX2lkIiBub3QgaW4gcm93IGZvciByb3cgaW4gYSArIGIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlYWwtdXNlciBjbHVzdGVycyBtdXN0IGJlIHN1cHBsaWVkIGZvciBhbGwgcGFpcmVkIGV2ZW50cyIpCiAgICBkZWx0YXMsIGdyb3VwcyA9IFtdLCB7fQogICAgZm9yIGksICh4LCB5KSBpbiBlbnVtZXJhdGUoemlwKGEsIGIsIHN0cmljdD1UcnVlKSk6CiAgICAgICAgaWYgY2x1c3RlcmVkIGFuZCB4WyJjbHVzdGVyX2lkIl0gIT0geVsiY2x1c3Rlcl9pZCJdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwYWlyZWQgdXNlciBjbHVzdGVycyBkbyBub3QgbWF0Y2giKQogICAgICAgIGRlbHRhID0gZmxvYXQoeFttZXRyaWNdKSAtIGZsb2F0KHlbbWV0cmljXSkKICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShkZWx0YSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBhaXJlZCBkZWx0YSBpcyBub3QgZmluaXRlIikKICAgICAgICBkZWx0YXMuYXBwZW5kKGRlbHRhKQogICAgICAgIGdyb3VwID0geFsiY2x1c3Rlcl9pZCJdIGlmIGNsdXN0ZXJlZCBlbHNlIGkKICAgICAgICB0b3RhbCwgY291bnQgPSBncm91cHMuZ2V0KGdyb3VwLCAoMC4wLCAwKSkKICAgICAgICBncm91cHNbZ3JvdXBdID0gKHRvdGFsICsgZGVsdGEsIGNvdW50ICsgMSkKICAgIGJsb2NrcyA9IGxpc3QoZ3JvdXBzLnZhbHVlcygpKQogICAgcm5nID0gcmFuZG9tLlJhbmRvbShzZWVkKQogICAgbWVhbnMgPSBbXQogICAgZm9yIF8gaW4gcmFuZ2Uobik6CiAgICAgICAgc2FtcGxlZCA9IFtibG9ja3Nbcm5nLnJhbmRyYW5nZShsZW4oYmxvY2tzKSldIGZvciBfIGluIGJsb2Nrc10KICAgICAgICBtZWFucy5hcHBlbmQoc3VtKHRvdGFsIGZvciB0b3RhbCwgXyBpbiBzYW1wbGVkKSAvIHN1bShjb3VudCBmb3IgXywgY291bnQgaW4gc2FtcGxlZCkpCiAgICBtZWFucy5zb3J0KCkKICAgIGFjdGl2ZSA9IFt0b3RhbCBmb3IgdG90YWwsIF8gaW4gYmxvY2tzIGlmIHRvdGFsICE9IDAuMF0KICAgIG9ic2VydmVkID0gYWJzKHN1bShkZWx0YXMpKQogICAgaWYgbGVuKGFjdGl2ZSkgPD0gMTY6CiAgICAgICAgbnVsbF90b3RhbHMgPSAoc3VtKHNpZ24gKiB0b3RhbCBmb3Igc2lnbiwgdG90YWwgaW4gemlwKHNpZ25zLCBhY3RpdmUpKQogICAgICAgICAgICAgICAgICAgICAgIGZvciBzaWducyBpbiBpdGVydG9vbHMucHJvZHVjdCgoLTEsIDEpLCByZXBlYXQ9bGVuKGFjdGl2ZSkpKQogICAgICAgIHBfdmFsdWUgPSBzdW0oYWJzKHRvdGFsKSA+PSBvYnNlcnZlZCAtIDFlLTEyIGZvciB0b3RhbCBpbiBudWxsX3RvdGFscykgLyAoMiAqKiBsZW4oYWN0aXZlKSkKICAgICAgICBwX21ldGhvZCA9ICJleGFjdCBwYWlyZWQgdXNlci1jbHVzdGVyIHNpZ24gZmxpcCIgaWYgY2x1c3RlcmVkIGVsc2UgImV4YWN0IHBhaXJlZCBldmVudCBzaWduIGZsaXAiCiAgICBlbHNlOgogICAgICAgIG51bGxfcm5nID0gcmFuZG9tLlJhbmRvbShzZWVkICsgMSkKICAgICAgICBleGNlZWQgPSBzdW0oCiAgICAgICAgICAgIGFicyhzdW0odG90YWwgaWYgbnVsbF9ybmcucmFuZHJhbmdlKDIpIGVsc2UgLXRvdGFsIGZvciB0b3RhbCBpbiBhY3RpdmUpKSA+PSBvYnNlcnZlZCAtIDFlLTEyCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKG4pCiAgICAgICAgKQogICAgICAgIHBfdmFsdWUgPSAoZXhjZWVkICsgMSkgLyAobiArIDEpCiAgICAgICAgcF9tZXRob2QgPSAiTW9udGUtQ2FybG8gcGFpcmVkIHVzZXItY2x1c3RlciBzaWduIGZsaXAiIGlmIGNsdXN0ZXJlZCBlbHNlICJNb250ZS1DYXJsbyBwYWlyZWQgZXZlbnQgc2lnbiBmbGlwIgogICAgcmV0dXJuIHsKICAgICAgICAibWV0cmljIjogbWV0cmljLAogICAgICAgICJldmVudHMiOiBsZW4oZGVsdGFzKSwKICAgICAgICAiaW5kZXBlbmRlbnRfY2x1c3RlcnMiOiBsZW4oYmxvY2tzKSwKICAgICAgICAicmVzYW1wbGluZ191bml0IjogInNvdXJjZV91c2VyIiBpZiBjbHVzdGVyZWQgZWxzZSAiZXZlbnQgKGlkZW50aXR5IG5vdCBzdXBwbGllZCkiLAogICAgICAgICJtZWFuX2RlbHRhIjogc3VtKGRlbHRhcykgLyBsZW4oZGVsdGFzKSwKICAgICAgICAiY2k5NSI6IFttZWFuc1tpbnQoMC4wMjUgKiBuKV0sIG1lYW5zW21pbihuIC0gMSwgaW50KDAuOTc1ICogbikpXV0sCiAgICAgICAgInBfdHdvX3NpZGVkIjogcF92YWx1ZSwKICAgICAgICAicF9tZXRob2QiOiBwX21ldGhvZCwKICAgICAgICAiY2hhbmdlZF9ldmVudHMiOiBzdW0oZGVsdGEgIT0gMCBmb3IgZGVsdGEgaW4gZGVsdGFzKSwKICAgICAgICAicG9zaXRpdmVfZXZlbnRzIjogc3VtKGRlbHRhID4gMCBmb3IgZGVsdGEgaW4gZGVsdGFzKSwKICAgICAgICAibmVnYXRpdmVfZXZlbnRzIjogc3VtKGRlbHRhIDwgMCBmb3IgZGVsdGEgaW4gZGVsdGFzKSwKICAgIH0KCgpkZWYgaG9sbV9hZGp1c3QocF92YWx1ZXM6IGRpY3Rbc3RyLCBmbG9hdF0pIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAiIiJIb2xtIHN0ZXAtZG93biBhZGp1c3RlZCBwLXZhbHVlcy4iIiIKICAgIG9yZGVyZWQgPSBzb3J0ZWQocF92YWx1ZXMuaXRlbXMoKSwga2V5PWxhbWJkYSBpdGVtOiBpdGVtWzFdKQogICAgYWRqdXN0ZWQ6IGRpY3Rbc3RyLCBmbG9hdF0gPSB7fQogICAgcnVubmluZyA9IDAuMAogICAgZm9yIHJhbmssIChuYW1lLCBwKSBpbiBlbnVtZXJhdGUob3JkZXJlZCk6CiAgICAgICAgcnVubmluZyA9IG1heChydW5uaW5nLCBtaW4oMS4wLCAobGVuKG9yZGVyZWQpIC0gcmFuaykgKiBwKSkKICAgICAgICBhZGp1c3RlZFtuYW1lXSA9IHJ1bm5pbmcKICAgIHJldHVybiBhZGp1c3RlZAoKCmRlZiByZXRyaWV2ZXJfdmVyZGljdChjb250cmFzdDogZGljdFtzdHIsIEFueV0pIC0+IHN0cjoKICAgICIiIkNsYXNzaWZ5IHJlcmFua2VyLW1pbnVzLXJldHJpZXZlciBORENHQDEwIHdpdGhvdXQgaGlkaW5nIGRlZ3JhZGF0aW9uLiIiIgogICAgbG93LCBoaWdoID0gY29udHJhc3RbImNpOTUiXQogICAgaWYgbG93ID49IE1JTl9NRUFOSU5HRlVMX05EQ0dfREVMVEE6CiAgICAgICAgaWYgY29udHJhc3QuZ2V0KCJwX3R3b19zaWRlZCIpIGlzIE5vbmUgb3IgY29udHJhc3RbInBfdHdvX3NpZGVkIl0gPiAwLjA1OgogICAgICAgICAgICByZXR1cm4gIlBPU0lUSVZFX0NJX05VTExfTk9UX1JFSkVDVEVEIgogICAgICAgIHJldHVybiAiQkVBVFNfUkVUUklFVkVSIgogICAgaWYgbG93ID4gMDoKICAgICAgICByZXR1cm4gIkdBSU5fQkVMT1dfTUVBTklOR0ZVTCIgaWYgaGlnaCA8IE1JTl9NRUFOSU5HRlVMX05EQ0dfREVMVEEgZWxzZSAiUE9TSVRJVkVfR0FJTl9NRUFOSU5HRlVMTkVTU19VTkNFUlRBSU4iCiAgICBpZiBoaWdoIDwgMDoKICAgICAgICByZXR1cm4gIldPUlNFX1RIQU5fUkVUUklFVkVSIgogICAgaWYgaGlnaCA8IE1JTl9NRUFOSU5HRlVMX05EQ0dfREVMVEE6CiAgICAgICAgcmV0dXJuICJOT19NRUFOSU5HRlVMX0dBSU4iCiAgICByZXR1cm4gIklOQ09OQ0xVU0lWRSIK", "experiment.json": "ewogICJzZWVkIjogMjAyNCwKICAidG9wX20iOiAyMCwKICAidHJhaW5fcGFpcnMiOiA1MzAsCiAgImV2YWxfdXNlcnMiOiAyNjUsCiAgInZhbGlkYXRpb25fdXNlcnMiOiAyNjUsCiAgImhpc3RvcnlfaXRlbXMiOiAzLAogICJiYXRjaF9zaXplIjogNCwKICAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIjogOCwKICAibGVhcm5pbmdfcmF0ZSI6IDAuMDAwMSwKICAic2Z0X2Vwb2NocyI6IDUsCiAgImRwb19lcG9jaHMiOiA1LAogICJkcm9wX2xhc3QiOiB0cnVlLAogICJwcmVjaXNpb24iOiAiYmYxNiIsCiAgIm1vZGVsX2lkIjogIlF3ZW4vUXdlbjIuNS1WTC0zQi1JbnN0cnVjdCIsCiAgIm1vZGVsX3JldmlzaW9uIjogIjY2Mjg1NTQ2ZDJiODIxY2Y0MjFkNGY1ZWIyNTc2MzU5ZDM3NzBjZDMiLAogICJ3ZWlnaHRzIjogWwogICAgMS4wLAogICAgMC4wLAogICAgMC41CiAgXSwKICAibm9pc2Vfc2lnbWEiOiAwLjA1LAogICJiZXRhMCI6IDAuMSwKICAiYmV0YV9mbG9vciI6IDFlLTA2LAogICJtYXhfcGl4ZWxzIjogMjYyMTQ0LAogICJidWRnZXRfc2Vjb25kcyI6IDY2MDAsCiAgInVwc3RyZWFtIjogewogICAgImxsbTJyZWMiOiB7CiAgICAgICJ1cmwiOiAiaHR0cHM6Ly9jb2RlbG9hZC5naXRodWIuY29tL0hhcHB5UG9pbnRlci9MTE0yUmVjL3ppcC83M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIiwKICAgICAgInNoYTI1NiI6ICI2YmNlZTdiMDllYzlhZDVmYTAxMDdhMDgzYzdiODA2MGFiYmVhZDIzZTEyNjRhNDk0ODA5MDZkZTYzOGMxY2M4IiwKICAgICAgImNvbW1pdCI6ICI3M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIgogICAgfSwKICAgICJoYW5vcmVjIjogewogICAgICAidXJsIjogImh0dHBzOi8vY29kZWxvYWQuZ2l0aHViLmNvbS93YW5neXUwNjI3L0hhTm9SZWMvemlwLzU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiLAogICAgICAic2hhMjU2IjogIjQ4YzUyZTU2MjVjZDRlMjVlODA2MDBmZWQ5YmVkOGFhZDg5MDY1NTZkMjY3Yjg3ZDUyODkwYjQ5YmYzM2Q2YTYiLAogICAgICAiY29tbWl0IjogIjU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiCiAgICB9CiAgfSwKICAiYXJ0aWZhY3RfcGlucyI6IHsKICAgICJkYXRhLnR4dCI6ICI0OGM3ZWFkOThhYmZhNzNjNTA2YmY3YWYxMzM3MDNmMmVlN2U5MjIxNjdkOTIyNzYwYmIwNzQzNzY2MzlhNzJjIiwKICAgICJ0cmFpbl9kYXRhLnR4dCI6ICIxMzAwZTVkZWVjMjlkNGJlZGU2ZTMyYmZhMWE1ZWFkZTU2M2Y1M2E2OTE2ZGE5NWQ2Nzc1NGVjNDBiOTQ3NWI4IiwKICAgICJ2YWxfZGF0YS50eHQiOiAiNDg0YTk3Y2ZiY2ZmMmE3OGU5MjYxMmIxNTg2M2NlMDhjODRjMjg1YzdiZmIzMjVlYzFmMzNkMzg4NGU2ZGUxNiIsCiAgICAidGVzdF9kYXRhLnR4dCI6ICI4MDA3NGZmZWEzNzkyOGQ5MmMzNTAzOGU5ZDhiYmYzZGUwY2ZjZjAxMzA4YWU0MTczZTNjNjgwYzFiYzUwZDFmIiwKICAgICJpdGVtX3RpdGxlcy5qc29uIjogIjE3ZjUwMTgwOWM0MTU5OTA1Y2ZhNmRmYTU2MjZkM2Q1MmQwYmUzMmJjOTc0N2VlODliMDAzNjYyZjZmNWU4NGMiLAogICAgIkxMTTJSZWMtYnVkZ2V0ZWQtfGthZ2dsZXx3b3JraW5nfExMTTJSZWN8cmVwZWF0ZWRfZXZhbHVhdGVfd2l0aF9zZXEtU2VwLTA5LTIwMjZfMTEtMTItMTMtYzkzZjEyLnB0aCI6ICJmYjBkMWQzOTE2ZjEwYjViNmYwZDlmNTk3ZTdiMmFjODQzN2Y0NDVjMTU2MjllYmYzMDk5MzMzYjcyODA2YTU5IiwKICAgICJRd2VuMi0wLjVCLUxMTTJSZWMtSUVNLWJ1ZGdldGVkX3N0ZXA1MDBfdGl0bGVfaXRlbV9lbWJzLm5weSI6ICI4OWY5ZDNkZDE3ZjllYzQ5NTM3ZjkxMGUzOWM5NzYxODc4Mjk5ZjdkMjRhYmRjZjgwNWFiMGRiOTE0MzNjMGYwIiwKICAgICJnYW1lc19zYXNyZWNfc3RlcDUwMC5sb2ciOiAiYTYwMTgxYTI2MTE0MDcyZGQ1NDQyZmQ5ZmZlNGIzMzg1MDUwMTBkOGIzMjFkOWUzMWNhMTY0NmIzNDFmZjE1NCIsCiAgICAiVmlkZW9fR2FtZXNfaW1hZ2VfbWFuaWZlc3QuanNvbmwiOiAiNWQ5YzU4NmQyZDA1YzZmZjIwOGM3YTBmOTQ1YTQ0MWE0ZmM2MGVmMGQxOGVjOGFhY2MyNjlmYmY2N2E0MmQwNCIKICB9LAogICJjaGVja3BvaW50X25hbWUiOiAiTExNMlJlYy1idWRnZXRlZC18a2FnZ2xlfHdvcmtpbmd8TExNMlJlY3xyZXBlYXRlZF9ldmFsdWF0ZV93aXRoX3NlcS1TZXAtMDktMjAyNl8xMS0xMi0xMy1jOTNmMTIucHRoIiwKICAiZW1iZWRkaW5nX25hbWUiOiAiUXdlbjItMC41Qi1MTE0yUmVjLUlFTS1idWRnZXRlZF9zdGVwNTAwX3RpdGxlX2l0ZW1fZW1icy5ucHkiLAogICJwdXJwb3NlIjogImNvcnJlY3RlZCBHYW1lcyBhZGFwdGF0aW9uOyBmdWxsIHNpZ25hbCByZXF1aXJlcyBQaGFzZSAzIHF1b3RhIGdhdGUiLAogICJpZGVudGl0eV9waW5zIjogewogICAgInRyYWluIjogIjg3ODE0ZmQ5NmI2ZDU5M2FmNmEyNjkxODUzN2I0YTQyYzI2ZGIxN2JjYWE2YTZiYzgyNGMxMTNmMzM1OWY0YWQiLAogICAgInZhbGlkYXRpb24iOiAiMDczOTNhMWRjNWEwNWM5Njc1MzJlODA1ODlkMWM5NGQ1NTEwNjBjZDc2Zjc3NjMxNGJkMzE0YzM0OTkyMTRhMiIsCiAgICAidGVzdCI6ICJmN2QyMTk1YWE0ODRmMTMzNWNjMGQ4OWMwZDI4NjMzZjU0OWJmYjNlYzBkY2Q0OTFjNmEzZTMyNDNkMjQwY2JlIgogIH0KfQo=", "audit.py": "IiIiRmFpbC1jbG9zZWQgYXVkaXRzIGZvciB0aGUgY29ycmVjdGVkIEhhTm9SZWMgR2FtZXMgc3R1ZHkuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKCmRlZiBfbG9hZChwYXRoOiBQYXRoKSAtPiBBbnk6CiAgICByZXR1cm4ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKCgpkZWYgcHJvdG9jb2xfZXJyb3JzKHByb3RvY29sOiBkaWN0W3N0ciwgQW55XSkgLT4gbGlzdFtzdHJdOgogICAgZXJyb3JzOiBsaXN0W3N0cl0gPSBbXQogICAgdGFzayA9IHByb3RvY29sLmdldCgidGFzayIsIHt9KQogICAgcmVjaXBlID0gcHJvdG9jb2wuZ2V0KCJyZWNpcGUiLCB7fSkKICAgIHNvdXJjZSA9IHByb3RvY29sLmdldCgic291cmNlX2F1dGhvcml0eSIsIHt9KQogICAgaWYgbm90IHRhc2suZ2V0KCJwcmltYXJ5X2NvbnRyYXN0cyIpOgogICAgICAgIGVycm9ycy5hcHBlbmQoInByaW1hcnkgY29udHJhc3RzIGFyZSBtaXNzaW5nIikKICAgIGlmIGZsb2F0KHRhc2suZ2V0KCJtZWFuaW5nZnVsX25kY2dfZGVsdGEiLCAwKSkgPD0gMDoKICAgICAgICBlcnJvcnMuYXBwZW5kKCJtZWFuaW5nZnVsIE5EQ0cgdGhyZXNob2xkIG11c3QgYmUgcG9zaXRpdmUiKQogICAgaWYgc291cmNlLmdldCgiaGFub3JlYyIsIHt9KS5nZXQoImNvbW1pdCIpICE9ICI1ODdmYWNlNzQ1MjRlNDU1M2I1YTdhYTI5NWZlOTYyMDA0NjgyMzgyIjoKICAgICAgICBlcnJvcnMuYXBwZW5kKCJIYU5vUmVjIHNvdXJjZSBjb21taXQgaXMgbm90IHBpbm5lZCIpCiAgICBpZiBzb3VyY2UuZ2V0KCJsbG0ycmVjIiwge30pLmdldCgiY29tbWl0IikgIT0gIjczYjQ4MWY3MTBmNjcxNjZhYjk1OGY0OTg1ZDI3YjI3ZmI0MTA4NzEiOgogICAgICAgIGVycm9ycy5hcHBlbmQoIkxMTTJSZWMgc291cmNlIGNvbW1pdCBpcyBub3QgcGlubmVkIikKICAgIGZvciBzdGFnZSBpbiAoInNmdCIsICJkcG8iKToKICAgICAgICBjb25maWcgPSByZWNpcGUuZ2V0KHN0YWdlLCB7fSkKICAgICAgICBmb3IgZmllbGQgaW4gKCJiYXRjaF9zaXplIiwgImdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcyIsICJlcG9jaHMiLCAibWF4X2dyYWRfbm9ybSIpOgogICAgICAgICAgICBpZiBmaWVsZCBub3QgaW4gY29uZmlnOgogICAgICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIntzdGFnZX0ue2ZpZWxkfSBpcyBtaXNzaW5nIikKICAgICAgICBpZiBzdGFnZSA9PSAic2Z0IiBhbmQgY29uZmlnLmdldCgicHJlY2lzaW9uIikgIT0gImJmMTYiOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKCJTRlQgcHJlY2lzaW9uIG11c3QgcmVtYWluIGJmMTYiKQogICAgaWYgcmVjaXBlLmdldCgiZHBvIiwge30pLmdldCgicmVmZXJlbmNlIikgIT0gImZyb3plbiBwb3N0LVNGVCBhZGFwdGVyLCBubyBEUE8gYWRhcHRlciI6CiAgICAgICAgZXJyb3JzLmFwcGVuZCgiRFBPIHJlZmVyZW5jZSBjb250cmFjdCBpcyBub3QgZnJvemVuIikKICAgIGlmIHByb3RvY29sLmdldCgic2VlZHMiKSAhPSBbMjAyNCwgMjAyNSwgMjAyNl06CiAgICAgICAgZXJyb3JzLmFwcGVuZCgidHJhaW5pbmcgc2VlZHMgbXVzdCBiZSBleGFjdGx5IDIwMjQsIDIwMjUsIDIwMjYiKQogICAgaWYgc29ydGVkKHByb3RvY29sLmdldCgid2VpZ2h0cyIsIFtdKSkgIT0gWzAuMCwgMC41LCAxLjBdOgogICAgICAgIGVycm9ycy5hcHBlbmQoIndlaWdodHMgbXVzdCBjb250YWluIDAsIDAuNSwgYW5kIDEiKQogICAgaWYgImludmFyaWFudHMiIG5vdCBpbiBwcm90b2NvbCBvciBsZW4ocHJvdG9jb2xbImludmFyaWFudHMiXSkgPCA4OgogICAgICAgIGVycm9ycy5hcHBlbmQoInByb3RvY29sIGludmFyaWFudHMgYXJlIGluY29tcGxldGUiKQogICAgc21va2UgPSBwcm90b2NvbC5nZXQoInNtb2tlIiwge30pCiAgICBpZiBub3Qgc21va2UuZ2V0KCJtdXN0X25vdF9iZV91c2VkX2FzX3NpZ25hbCIsIEZhbHNlKToKICAgICAgICBlcnJvcnMuYXBwZW5kKCJzbW9rZSBtdXN0IGJlIGV4cGxpY2l0bHkgbm9uLXNpZ25hbCIpCiAgICByZXR1cm4gZXJyb3JzCgoKZGVmIGF1ZGl0X3Byb3RvY29sKHBhdGg6IFBhdGgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgcHJvdG9jb2wgPSBfbG9hZChwYXRoKQogICAgZXJyb3JzID0gcHJvdG9jb2xfZXJyb3JzKHByb3RvY29sKQogICAgcmVzdWx0ID0geyJzdGF0dXMiOiAiUEFTUyIgaWYgbm90IGVycm9ycyBlbHNlICJGQUlMIiwgImVycm9ycyI6IGVycm9ycywgInByb3RvY29sIjogc3RyKHBhdGgpfQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yKSkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgX2ZpbmQocm9vdDogUGF0aCwgbmFtZTogc3RyKSAtPiBsaXN0W1BhdGhdOgogICAgcmV0dXJuIHNvcnRlZChwYXRoIGZvciBwYXRoIGluIHJvb3Qucmdsb2IobmFtZSkgaWYgcGF0aC5pc19maWxlKCkpCgoKZGVmIGF1ZGl0X2FydGlmYWN0cyhyb290OiBQYXRoLCBwcm90b2NvbDogZGljdFtzdHIsIEFueV0sICosIHBhcmVudF9yb290OiBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgcmVxdWlyZWQgPSBbInNmdF9idW5kbGUuanNvbiIsICJzZnRfbG9yYV9zdGF0ZS5wdCJdCiAgICBtaXNzaW5nID0gW25hbWUgZm9yIG5hbWUgaW4gcmVxdWlyZWQgaWYgbm90IF9maW5kKHBhcmVudF9yb290IG9yIHJvb3QsIG5hbWUpXQogICAgcGFyZW50X2hhc2hlcyA9IHNldCgpCiAgICBmb3IgcGF0aCBpbiBfZmluZChwYXJlbnRfcm9vdCBvciByb290LCAic2Z0X2xvcmFfc3RhdGUucHQiKToKICAgICAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIHBhcmVudF9oYXNoZXMuYWRkKGhhc2hsaWIuZmlsZV9kaWdlc3QoaGFuZGxlLCAic2hhMjU2IikuaGV4ZGlnZXN0KCkpCiAgICBhcm1fcmVzdWx0cyA9IF9maW5kKHJvb3QsICJhcm1fcmVzdWx0Lmpzb24iKQogICAgZXJyb3JzID0gW2YibWlzc2luZyB7bmFtZX0iIGZvciBuYW1lIGluIG1pc3NpbmddCiAgICBleHBlY3RlZCA9ICgKICAgICAgICB7KGZsb2F0KGFybVsid2VpZ2h0Il0pLCBhcm1bImltYWdlX2NvbmRpdGlvbiJdLCBhcm0uZ2V0KCJoYXJkbmVzc19jb250cm9sIikpCiAgICAgICAgIGZvciBhcm0gaW4gcHJvdG9jb2xbImV4ZWN1dGlvbl9tYXRyaXgiXX0KICAgICAgICBpZiAiZXhlY3V0aW9uX21hdHJpeCIgaW4gcHJvdG9jb2wgZWxzZQogICAgICAgIHsoZmxvYXQod2VpZ2h0KSwgY29uZGl0aW9uLCBOb25lKQogICAgICAgICBmb3Igd2VpZ2h0IGluIHByb3RvY29sWyJ3ZWlnaHRzIl0gZm9yIGNvbmRpdGlvbiBpbiBwcm90b2NvbFsiY29udHJvbHMiXVsiaW1hZ2VfY29uZGl0aW9ucyJdfQogICAgKQogICAgc21va2VfbWFuaWZlc3QgPSByb290IC8gInNtb2tlX21hbmlmZXN0Lmpzb24iCiAgICBpZiBzbW9rZV9tYW5pZmVzdC5leGlzdHMoKToKICAgICAgICBzbW9rZSA9IF9sb2FkKHNtb2tlX21hbmlmZXN0KQogICAgICAgIGlmIHNtb2tlLmdldCgibXVzdF9ub3RfYmVfdXNlZF9hc19zaWduYWwiKSBpcyBub3QgVHJ1ZToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZCgic21va2UgbWFuaWZlc3QgaXMgbm90IGV4cGxpY2l0bHkgbm9uLXNpZ25hbCIpCiAgICAgICAgaWYgc21va2UuZ2V0KCJwdXJwb3NlIikgIT0gImNvcnJlY3RuZXNzX29ubHlfbm9uX3NpZ25hbCI6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoInNtb2tlIHB1cnBvc2UgaXMgbm90IGNvcnJlY3RuZXNzLW9ubHkiKQogICAgICAgIGV4cGVjdGVkID0geygxLjAsICJyZWFsIiwgTm9uZSl9CiAgICAgICAgZm9yIGxhYmVsIGluICgic2Z0X2dwdV9tZW1vcnkiLCAiZHBvX2dwdV9tZW1vcnkiKToKICAgICAgICAgICAgbWVtb3J5ID0gc21va2UuZ2V0KGxhYmVsLCB7fSkKICAgICAgICAgICAgdG90YWwgPSBpbnQobWVtb3J5LmdldCgidG90YWxfYnl0ZXMiLCAwKSkKICAgICAgICAgICAgYWxsb2NhdGVkID0gaW50KG1lbW9yeS5nZXQoInBlYWtfYWxsb2NhdGVkX2J5dGVzIiwgMCkpCiAgICAgICAgICAgIHJlc2VydmVkID0gaW50KG1lbW9yeS5nZXQoInBlYWtfcmVzZXJ2ZWRfYnl0ZXMiLCAwKSkKICAgICAgICAgICAgaWYgbm90ICgwIDwgYWxsb2NhdGVkIDw9IHJlc2VydmVkIDw9IHRvdGFsKToKICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJpbnZhbGlkIHtsYWJlbH0gdGVsZW1ldHJ5IikKICAgICAgICBmb3IgbGFiZWwsIHByZWZpeCBpbiAoKCJzZnRfc3RhZ2VfdGltaW5ncyIsICJzZnRfdXBkYXRlIiksICgiZHBvX3N0YWdlX3RpbWluZ3MiLCAiZHBvX3VwZGF0ZV8iKSk6CiAgICAgICAgICAgIHRpbWluZ3MgPSBzbW9rZS5nZXQobGFiZWwsIFtdKQogICAgICAgICAgICB1cGRhdGVzID0gWwogICAgICAgICAgICAgICAgcm93IGZvciByb3cgaW4gdGltaW5ncwogICAgICAgICAgICAgICAgaWYgc3RyKHJvdy5nZXQoInN0YWdlIiwgIiIpKS5zdGFydHN3aXRoKHByZWZpeCkKICAgICAgICAgICAgXQogICAgICAgICAgICBpZiBub3QgdXBkYXRlcyBvciBhbnkoZmxvYXQocm93LmdldCgic2Vjb25kcyIsIDApKSA8PSAwIGZvciByb3cgaW4gdXBkYXRlcyk6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiaW52YWxpZCB7bGFiZWx9IG9wdGltaXplciB0aW1pbmciKQogICAgb2JzZXJ2ZWQ6IHNldFt0dXBsZVtmbG9hdCwgc3RyLCBzdHIgfCBOb25lXV0gPSBzZXQoKQogICAgZm9yIHBhdGggaW4gYXJtX3Jlc3VsdHM6CiAgICAgICAgcmVzdWx0ID0gX2xvYWQocGF0aCkKICAgICAgICBrZXkgPSAoZmxvYXQocmVzdWx0LmdldCgid2VpZ2h0IiwgZmxvYXQoIm5hbiIpKSksIHN0cihyZXN1bHQuZ2V0KCJpbWFnZV9jb25kaXRpb24iLCAiIikpLAogICAgICAgICAgICAgICByZXN1bHQuZ2V0KCJoYXJkbmVzc19jb250cm9sIikpCiAgICAgICAgaWYga2V5IGluIG9ic2VydmVkOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiZHVwbGljYXRlIGFybToge2tleX0iKQogICAgICAgIG9ic2VydmVkLmFkZChrZXkpCiAgICAgICAgaWYgcmVzdWx0LmdldCgic3RhdHVzIikgIT0gIkNPTVBMRVRFIjoKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmImluY29tcGxldGUgYXJtOiB7cGF0aH0iKQogICAgICAgIHByZWRpY3Rpb25zID0gcmVzdWx0LmdldCgidmFsaWRhdGlvbl9wcmVkaWN0aW9ucyIgaWYgcHJvdG9jb2wuZ2V0KCJldmFsdWF0aW9uX3NwbGl0IikgPT0gInZhbGlkYXRpb24iIGVsc2UgInByZWRpY3Rpb25zIiwgW10pCiAgICAgICAgaWYgbm90IHByZWRpY3Rpb25zOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiYXJtIGhhcyBubyBwcmVkaWN0aW9uczoge3BhdGh9IikKICAgICAgICBmb3Igcm93IGluIHByZWRpY3Rpb25zOgogICAgICAgICAgICBmb3IgbWV0cmljIGluICgibmRjZ0AxMCIsICJyZWNhbGxAMTAiLCAiY2FuZGlkYXRlX3JlY2FsbEAyMCIpOgogICAgICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoZmxvYXQocm93LmdldChtZXRyaWMsIGZsb2F0KCJuYW4iKSkpKToKICAgICAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYibm9uLWZpbml0ZSB7bWV0cmljfToge3BhdGh9IikKICAgICAgICBjaGVja3BvaW50ID0gcmVzdWx0LmdldCgiY2hlY2twb2ludCIpCiAgICAgICAgY2hlY2twb2ludF9wYXRoID0gKAogICAgICAgICAgICBwYXRoLnBhcmVudCAvIGNoZWNrcG9pbnQKICAgICAgICAgICAgaWYgY2hlY2twb2ludCBhbmQgbm90IFBhdGgoY2hlY2twb2ludCkuaXNfYWJzb2x1dGUoKQogICAgICAgICAgICBlbHNlIFBhdGgoY2hlY2twb2ludCkgaWYgY2hlY2twb2ludCBlbHNlIE5vbmUKICAgICAgICApCiAgICAgICAgaWYgY2hlY2twb2ludF9wYXRoIGlzIE5vbmUgb3Igbm90IGNoZWNrcG9pbnRfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm1pc3NpbmcgY2hlY2twb2ludDoge3BhdGh9IikKICAgICAgICBpZiBub3QgcmVzdWx0LmdldCgicGFyZW50X3NmdF9zaGEyNTYiKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm1pc3NpbmcgcGFyZW50IGhhc2g6IHtwYXRofSIpCiAgICAgICAgaWYgcGFyZW50X2hhc2hlcyBhbmQgcmVzdWx0LmdldCgicGFyZW50X3NmdF9zaGEyNTYiKSBub3QgaW4gcGFyZW50X2hhc2hlczoKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmInBhcmVudCBjaGVja3BvaW50IGhhc2ggbWlzbWF0Y2g6IHtwYXRofSIpCiAgICBpZiBvYnNlcnZlZCAhPSBleHBlY3RlZDoKICAgICAgICBlcnJvcnMuYXBwZW5kKGYiYXJtIG1hdHJpeCBtaXNtYXRjaDogZXhwZWN0ZWQge3NvcnRlZChleHBlY3RlZCwga2V5PXJlcHIpfSwgb2JzZXJ2ZWQge3NvcnRlZChvYnNlcnZlZCwga2V5PXJlcHIpfSIpCiAgICBvdXRwdXQgPSB7CiAgICAgICAgInN0YXR1cyI6ICJQQVNTIiBpZiBub3QgZXJyb3JzIGVsc2UgIkZBSUwiLAogICAgICAgICJlcnJvcnMiOiBlcnJvcnMsCiAgICAgICAgInNmdF9idW5kbGVfY291bnQiOiBsZW4oX2ZpbmQocGFyZW50X3Jvb3Qgb3Igcm9vdCwgInNmdF9idW5kbGUuanNvbiIpKSwKICAgICAgICAiYXJtX3Jlc3VsdF9jb3VudCI6IGxlbihhcm1fcmVzdWx0cyksCiAgICAgICAgImV4cGVjdGVkX2FybV9jb3VudCI6IGxlbihleHBlY3RlZCksCiAgICB9CiAgICBwcmludChqc29uLmR1bXBzKG91dHB1dCwgaW5kZW50PTIpKQogICAgcmV0dXJuIG91dHB1dAoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgc3VicGFyc2VycyA9IHBhcnNlci5hZGRfc3VicGFyc2VycyhkZXN0PSJjb21tYW5kIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHByb3RvY29sX3BhcnNlciA9IHN1YnBhcnNlcnMuYWRkX3BhcnNlcigicHJvdG9jb2wiKQogICAgcHJvdG9jb2xfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcm90b2NvbCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFydGlmYWN0c19wYXJzZXIgPSBzdWJwYXJzZXJzLmFkZF9wYXJzZXIoImFydGlmYWN0cyIpCiAgICBhcnRpZmFjdHNfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcm90b2NvbCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFydGlmYWN0c19wYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBhcnRpZmFjdHNfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wYXJlbnQtcm9vdCIsIHR5cGU9UGF0aCwgaGVscD0ic2VwYXJhdGUgbW91bnRlZCBTRlQgcGFyZW50IGRpcmVjdG9yeSIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgaWYgYXJncy5jb21tYW5kID09ICJwcm90b2NvbCI6CiAgICAgICAgcmVzdWx0ID0gYXVkaXRfcHJvdG9jb2woYXJncy5wcm90b2NvbCkKICAgIGVsc2U6CiAgICAgICAgcHJvdG9jb2xfcmVzdWx0ID0gYXVkaXRfcHJvdG9jb2woYXJncy5wcm90b2NvbCkKICAgICAgICBpZiBwcm90b2NvbF9yZXN1bHRbInN0YXR1cyJdICE9ICJQQVNTIjoKICAgICAgICAgICAgcmV0dXJuIDEKICAgICAgICByZXN1bHQgPSBhdWRpdF9hcnRpZmFjdHMoYXJncy5yb290LCBfbG9hZChhcmdzLnByb3RvY29sKSwgcGFyZW50X3Jvb3Q9YXJncy5wYXJlbnRfcm9vdCkKICAgIHJldHVybiAwIGlmIHJlc3VsdFsic3RhdHVzIl0gaW4geyJQQVNTIiwgIkNPTVBMRVRFIn0gZWxzZSAxCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo=", "run-protocol.json": "ewogICJwcm90b2NvbF92ZXJzaW9uIjogMywKICAicGxhbiI6ICIyNjA5MjAtMTI0NS1oYW5vcmVjLWZhaXRoZnVsLXNpZ25hbC1yZWNoZWNrIiwKICAidGFzayI6IHsKICAgICJkb21haW4iOiAiR2FtZXMgYWRhcHRhdGlvbiBvZiBIYU5vUmVjIGhpdC0xIGJpbmFyeSBwcmVmZXJlbmNlIiwKICAgICJuYXRpdmVfc2NvcmVfY2xhaW0iOiBmYWxzZSwKICAgICJwcmltYXJ5X21ldHJpY3MiOiBbCiAgICAgICJORENHQDEwIiwKICAgICAgIlJlY2FsbEAxMCIKICAgIF0sCiAgICAicHJpbWFyeV9jb250cmFzdHMiOiBbCiAgICAgICJ3PTAuMCBtaW51cyB3PTEuMCIsCiAgICAgICJ3PTAuNSBtaW51cyB3PTEuMCIKICAgIF0sCiAgICAibWVhbmluZ2Z1bF9uZGNnX2RlbHRhIjogMC4wMDIsCiAgICAiY29uZmlybWF0aW9uX3NwbGl0IjogInRlc3QiLAogICAgImRldmVsb3BtZW50X3NwbGl0IjogInZhbGlkYXRpb24iLAogICAgImNhbmRpZGF0ZV9zb3VyY2UiOiAiZnJvemVuIFNBU1JlYyB0b3AtTSBjYW5kaWRhdGVzOyB0YXJnZXQgbmV2ZXIgaW5zZXJ0ZWQiLAogICAgInNjb3JpbmciOiAic2luZ2xlIGFuc3dlci10b2tlbiBsb2cgUChZZXMpLWxvZyBQKE5vKSBhdCB0aGUgY2hhdCBnZW5lcmF0aW9uIHByZWZpeDsgbm8gYXNzaXN0YW50IHRlcm1pbmF0b3IvRU9TIGluIHRoZSBzY29yZSIKICB9LAogICJzb3VyY2VfYXV0aG9yaXR5IjogewogICAgImhhbm9yZWMiOiB7CiAgICAgICJyZXBvc2l0b3J5IjogImh0dHBzOi8vZ2l0aHViLmNvbS93YW5neXUwNjI3L0hhTm9SZWMiLAogICAgICAiY29tbWl0IjogIjU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiLAogICAgICAiYXJjaGl2ZV9zaGEyNTYiOiAiNDhjNTJlNTYyNWNkNGUyNWU4MDYwMGZlZDliZWQ4YWFkODkwNjU1NmQyNjdiODdkNTI4OTBiNDliZjMzZDZhNiIsCiAgICAgICJwYXRocyI6IFsKICAgICAgICAiY29uZmlncy9oYW5vcmVjL21pY3JvbGVuc19oaXQxLnlhbWwiLAogICAgICAgICJoYW5vcmVjL3RyYWluZXIvZHBvLnB5IiwKICAgICAgICAiaGFub3JlYy9oYXJzL21hdGgucHkiLAogICAgICAgICJoYW5vcmVjL2hhcnMvaGFyZG5lc3MucHkiLAogICAgICAgICJoYW5vcmVjL25vZG8vaG9va3MucHkiCiAgICAgIF0KICAgIH0sCiAgICAibGxhbWFmYWN0b3J5IjogewogICAgICAicmVxdWlyZW1lbnQiOiAibGxhbWFmYWN0b3J5PT0wLjkuMy5kZXYwIiwKICAgICAgInNvdXJjZV9yZXZpc2lvbiI6IG51bGwsCiAgICAgICJub3RlIjogIlRoZSBwaW5uZWQgSGFOb1JlYyByZXF1aXJlbWVudCBleHBvc2VzIGEgcGFja2FnZSB2ZXJzaW9uIGJ1dCBubyBzb3VyY2UgY29tbWl0LiBUaGUgY29ycmVjdGVkIEdhbWVzIHJ1bm5lciBwb3J0cyBhbmQgdGVzdHMgdGhlIG9ic2VydmFibGUgSGFOb1JlYyB0cmFpbmVyIGNvbnRyYWN0OyBpdCBkb2VzIG5vdCBjbGFpbSBieXRlLWlkZW50aWNhbCBMTGFNQS1GYWN0b3J5IGludGVybmFscy4iCiAgICB9LAogICAgImxsbTJyZWMiOiB7CiAgICAgICJyZXBvc2l0b3J5IjogImh0dHBzOi8vZ2l0aHViLmNvbS9IYXBweVBvaW50ZXIvTExNMlJlYyIsCiAgICAgICJjb21taXQiOiAiNzNiNDgxZjcxMGY2NzE2NmFiOTU4ZjQ5ODVkMjdiMjdmYjQxMDg3MSIsCiAgICAgICJhcmNoaXZlX3NoYTI1NiI6ICI2YmNlZTdiMDllYzlhZDVmYTAxMDdhMDgzYzdiODA2MGFiYmVhZDIzZTEyNjRhNDk0ODA5MDZkZTYzOGMxY2M4IgogICAgfQogIH0sCiAgInJlY2lwZSI6IHsKICAgICJtb2RlbCI6ICJRd2VuL1F3ZW4yLjUtVkwtM0ItSW5zdHJ1Y3QiLAogICAgIm1vZGVsX3JldmlzaW9uIjogIjY2Mjg1NTQ2ZDJiODIxY2Y0MjFkNGY1ZWIyNTc2MzU5ZDM3NzBjZDMiLAogICAgInNmdCI6IHsKICAgICAgImFkYXB0ZXIiOiAidGFzay1zcGVjaWZpYyBMb1JBIGFkYXB0ZXIiLAogICAgICAiYmF0Y2hfc2l6ZSI6IDIsCiAgICAgICJncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMiOiA4LAogICAgICAiZXBvY2hzIjogNS4wLAogICAgICAic2NoZWR1bGVyIjogImNvc2luZSIsCiAgICAgICJ3YXJtdXBfc3RlcHMiOiAwLAogICAgICAibWF4X2dyYWRfbm9ybSI6IDEuMCwKICAgICAgInByZWNpc2lvbiI6ICJiZjE2IiwKICAgICAgImxvcmFfcmFuayI6IDgsCiAgICAgICJsb3JhX2FscGhhIjogMzIsCiAgICAgICJsb3JhX2Ryb3BvdXQiOiAwLjA1LAogICAgICAidGFyZ2V0X21vZHVsZXMiOiAibGFuZ3VhZ2UtbW9kZWwgbGluZWFyIGxheWVycyAocSxrLHYsbyxnYXRlLHVwLGRvd24pOyB2aXNpb24gdG93ZXIgZnJvemVuIiwKICAgICAgInRhc2siOiAiNTAlIFllcyAvIDUwJSBObyIKICAgIH0sCiAgICAiZHBvIjogewogICAgICAiZnJlc2hfYWRhcHRlciI6IHRydWUsCiAgICAgICJyZWZlcmVuY2UiOiAiZnJvemVuIHBvc3QtU0ZUIGFkYXB0ZXIsIG5vIERQTyBhZGFwdGVyIiwKICAgICAgImJhdGNoX3NpemUiOiA0LAogICAgICAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIjogOCwKICAgICAgImVwb2NocyI6IDUuMCwKICAgICAgInNjaGVkdWxlciI6ICJjb3NpbmUiLAogICAgICAid2FybXVwX3N0ZXBzIjogMCwKICAgICAgIm1heF9ncmFkX25vcm0iOiAxLjAsCiAgICAgICJiZXRhMCI6IDAuMSwKICAgICAgImxvc3MiOiAic2lnbW9pZCIsCiAgICAgICJub2lzZV9zaWdtYSI6IDAuMDUsCiAgICAgICJiZXRhX2Zsb29yIjogMWUtMDYsCiAgICAgICJkcm9wX2xhc3QiOiB0cnVlLAogICAgICAiZHJvcG91dCI6ICJkaXNhYmxlZCIsCiAgICAgICJ0YXJnZXRfbW9kdWxlcyI6ICJsYW5ndWFnZS1tb2RlbCBsaW5lYXIgbGF5ZXJzIgogICAgfQogIH0sCiAgInNlZWRzIjogWwogICAgMjAyNCwKICAgIDIwMjUsCiAgICAyMDI2CiAgXSwKICAid2VpZ2h0cyI6IFsKICAgIDEuMCwKICAgIDAuMCwKICAgIDAuNQogIF0sCiAgImNvbnRyb2xzIjogewogICAgImltYWdlX2NvbmRpdGlvbnMiOiBbCiAgICAgICJyZWFsIiwKICAgICAgInNodWZmbGUiCiAgICBdLAogICAgInNodWZmbGUiOiAiZml4ZWQtcG9pbnQtZnJlZSBmcmVxdWVuY3ktc3RyYXRpZmllZCBpdGVtIGJpamVjdGlvbiBidWlsdCBmcm9tIHRyYWluaW5nIHJvd3Mgb25seSIsCiAgICAiY29uc3RhbnRfaGFyZG5lc3MiOiAidmFsaWRhdGlvbi1vbmx5IG1lYW4tbWF0Y2hlZCBkaWFnbm9zdGljOyBub3QgYSBwcmltYXJ5IHRlc3QgYXJtIgogIH0sCiAgInNtb2tlIjogewogICAgInB1cnBvc2UiOiAiY29ycmVjdG5lc3MgYW5kIG1lYXN1cmVkIHJlc291cmNlIHNtb2tlIG9ubHkiLAogICAgInRyYWluX3BhaXJzIjogOCwKICAgICJ2YWxpZGF0aW9uX3VzZXJzIjogNCwKICAgICJ0ZXN0X3VzZXJzIjogNCwKICAgICJlcG9jaHMiOiAxLAogICAgIm1heF9vcHRpbWl6ZXJfdXBkYXRlcyI6IDIsCiAgICAibXVzdF9ub3RfYmVfdXNlZF9hc19zaWduYWwiOiB0cnVlCiAgfSwKICAiaW52YXJpYW50cyI6IFsKICAgICJ0cmFpbmluZyBwYWlyIGFuZCBoYXJkbmVzcyBoYXNoZXMgZG8gbm90IGNoYW5nZSB3aGVuIGhlbGQtb3V0IHRhcmdldHMvY2FuZGlkYXRlcy9pbWFnZXMgY2hhbmdlIiwKICAgICJuZWdhdGl2ZSBpcyBub3QgdGFyZ2V0LCBvYnNlcnZlZCBoaXN0b3J5LCBvciB0cmFpbi1rbm93biBmdXR1cmUgaXRlbSIsCiAgICAic2h1ZmZsZSBpcyBhIGJpamVjdGlvbiBhbmQgaGFzIG5vIGZpeGVkIHBvaW50cyIsCiAgICAiZ2xvYmFsIHJlc3BvbnNpdmVuZXNzIGJhdGNoIGhhcyBhdCBsZWFzdCB0aHJlZSBleGFtcGxlcyIsCiAgICAib25seSB0aGUgZnJlc2ggRFBPIGFkYXB0ZXIgY2hhbmdlcyBkdXJpbmcgRFBPIiwKICAgICJTRlQtb25seSByZWZlcmVuY2UgaXMgaW52YXJpYW50IHRvIERQTyB1cGRhdGVzIGFuZCBOb0RPIG5vaXNlIiwKICAgICJwYXJ0aWFsIG9yIG1pc3Npbmcgc3RhZ2VzIGNhbm5vdCBiZSBsYWJlbGxlZCBDT01QTEVURSIsCiAgICAic291cmNlL2NvbmZpZy9kYXRhL3BhcmVudCBoYXNoZXMgYXJlIHBlcnNpc3RlZCBhbmQgY2hlY2tlZCIKICBdLAogICJzdGF0aXN0aWNzIjogewogICAgImJvb3RzdHJhcF9yZXNhbXBsZXMiOiAyMDAwMCwKICAgICJib290c3RyYXBfc2VlZCI6IDIwMjYxMDAxLAogICAgImhvbG1fcHJpbWFyeV9jb250cmFzdHMiOiAyLAogICAgInRyYWluaW5nX3NlZWRfdW5jZXJ0YWludHkiOiAicmVwb3J0IGluZGl2aWR1YWwgc2VlZHMgYW5kIHNwcmVhZDsgYm9vdHN0cmFwIGlzIGNvbmRpdGlvbmFsIG9uIGZpdHRlZCBzZWVkIG1vZGVscyIsCiAgICAicF92YWx1ZSI6ICJwYWlyZWQgd2hvbGUtdXNlciBzaWduLWZsaXAgbnVsbCBkaXN0cmlidXRpb24gKGV4YWN0IDw9MTYgbm9uemVybyBjbHVzdGVycywgb3RoZXJ3aXNlIE1vbnRlIENhcmxvICsxKTsgZXhjaGFuZ2VhYmlsaXR5IGFzc3VtcHRpb24iLAogICAgInJlc2FtcGxpbmdfdW5pdCI6ICJ0cnVlIHNvdXJjZS11c2VyIGNsdXN0ZXI7IGV2ZW50LXdlaWdodGVkIG1lYW4iLAogICAgIm1lYW5pbmdmdWxfZ2FpbiI6ICJsb3dlciBDSSBib3VuZCA+PSswLjAwMiBhbmQgbnVsbCBwPD0wLjA1OyBwcmltYXJ5IGZhbWlseSB1c2VzIEhvbG07IHNlY29uZGFyeSBjb250cm9scyBleHBsaWNpdGx5IG1hcmtlZCIKICB9LAogICJyZXBhaXJzX3YyIjogWwogICAgIlNGVCB0YXNrIGlzIHRoZSBIYU5vUmVjIGhpdD0xIGJpbmFyeSB0YXNrOiB0cnVlIG5leHQgaXRlbSAtPiBZZXMsIHNlZWRlZCByYW5kb20gdHJhaW4tb25seSB1bnNlZW4gaXRlbSAtPiBObyAoNTAvNTApOyBoYXJkIENGIG5lZ2F0aXZlcyBhcmUgRFBPLW9ubHkiLAogICAgIkRQTyBwb2xpY3kgcnVucyB3aXRoIGFsbCBkcm9wb3V0IG1vZHVsZXMgYXQgcD0wIChMTGFNQS1GYWN0b3J5IGRpc2FibGVfZHJvcG91dF9pbl9tb2RlbCk7IHJlZmVyZW5jZSBpcyB0aGUgU0ZUIGFkYXB0ZXIgb25seSIsCiAgICAiTG9SQSB0YXJnZXRzIGxhbmd1YWdlLW1vZGVsIGxpbmVhciBsYXllcnMgb25seTsgdmlzaW9uIHRvd2VyIGFuZCBtZXJnZXIgZnJvemVuIChMTGFNQS1GYWN0b3J5IGZyZWV6ZV92aXNpb25fdG93ZXI9VHJ1ZSkiLAogICAgImNhbmRpZGF0ZSBzY29yZSB0aWVzIGtlZXAgdGhlIGZyb3plbiBTQVNSZWMgb3JkZXIgaW5zdGVhZCBvZiBpdGVtLWlkIG9yZGVyIiwKICAgICJTRlQgbWljcm8tYmF0Y2ggMiB4IGFjY3VtdWxhdGlvbiA4IGZvbGxvd3MgdHJhaW4vbWxsbV9zZnQuc2giLAogICAgIm5hdGl2ZSBiZjE2IGlzIHJlcXVpcmVkIGJ5IHRoZSBvcmlnaW5hbCBwcm90b2NvbDsgS2FnZ2xlIHNlcnZlcyBvbmx5IFQ0L1AxMDAgKGJmMTYgZW11bGF0ZWQgb24gVDQpIHNvIHRoaXMgaXMgcmVjb3JkZWQgYXMgYW4gYWRhcHRhdGlvbiwgbm90IGVuZm9yY2VkIiwKICAgICJTRlQgdXNlcyBib3RoIGEgWWVzIGFuZCBhIE5vIHNhbXBsZSBwZXIgdHJhaW5pbmcgcGFpciAoMnggZXhhbXBsZXMvZXBvY2ggdnMgdXBzdHJlYW0ncyBvbmUgcmFuZG9tbHktbGFiZWxsZWQgc2FtcGxlIHBlciByb3cpOyBlcG9jaHMgYW5kIG1heF9waXhlbHMgYXJlIGRlcml2ZWQgZnJvbSBtZWFzdXJlZCBUNCBzcGVlZC9WUkFNIGFuZCByZWNvcmRlZCBhcyBhZGFwdGF0aW9ucyIKICBdLAogICJpZGVudGl0eV9waW5zIjogewogICAgInRyYWluIjogIjg3ODE0ZmQ5NmI2ZDU5M2FmNmEyNjkxODUzN2I0YTQyYzI2ZGIxN2JjYWE2YTZiYzgyNGMxMTNmMzM1OWY0YWQiLAogICAgInZhbGlkYXRpb24iOiAiMDczOTNhMWRjNWEwNWM5Njc1MzJlODA1ODlkMWM5NGQ1NTEwNjBjZDc2Zjc3NjMxNGJkMzE0YzM0OTkyMTRhMiIsCiAgICAidGVzdCI6ICJmN2QyMTk1YWE0ODRmMTMzNWNjMGQ4OWMwZDI4NjMzZjU0OWJmYjNlYzBkY2Q0OTFjNmEzZTMyNDNkMjQwY2JlIgogIH0sCiAgInJlcGFpcnNfdjMiOiBbCiAgICAiSGFzaC1waW5uZWQgc291cmNlIENTViwgdmVyaWZpZWQgQ1NWIGl0ZW0gSURzICsxIGVxdWFscyBkb3duc3RyZWFtIHJvdyBzZXF1ZW5jZS4gVXNlciBpZGVudGlmaWVycyBwc2V1ZG9ueW1pemVkLiIsCiAgICAiTmVnYXRpdmVzIGV4Y2x1ZGUgZXZlcnkgVFJBSU4ta25vd24gaXRlbSBvZiB0aGUgc2FtZSB0cnVlIHVzZXIsIGluY2x1ZGluZyBoaXN0b3J5IG91dHNpZGUgdGhlIDEwLWl0ZW0gd2luZG93LiBOZXZlciBjb25zdWx0IGhlbGQtb3V0IGxhYmVscy4iLAogICAgIlByZWRpY3Rpb24gZXZlbnQgSURzIHN0YXkgc3RhYmxlOyBjbHVzdGVyX2lkIGNhcnJpZXMgdHJ1ZS11c2VyIGdyb3VwaW5nLiBDbHVzdGVyIGJvb3RzdHJhcCBhbmQgcGFpcmVkIG51bGwgcGVybXV0YXRpb24gcmVwbGFjZSByb3ctaW5kZXBlbmRlbnQgaW5mZXJlbmNlLiIsCiAgICAiSGlzdG9yaWNhbCByZXBhaXJlZCBTRlQgdjIvRFBPIGFybXMgdjEgcmVtYWluIGltbXV0YWJsZSBhbmQgbWVjaGFuaXNtLWluZWxpZ2libGUgZHVlIHRvIGtub3duLXBvc2l0aXZlIG5lZ2F0aXZlcy4iCiAgXSwKICAiZXhlY3V0aW9uX21hdHJpeCI6IFsKICAgIHsKICAgICAgIndlaWdodCI6IDEuMCwKICAgICAgImltYWdlX2NvbmRpdGlvbiI6ICJyZWFsIiwKICAgICAgImhhcmRuZXNzX2NvbnRyb2wiOiBudWxsCiAgICB9LAogICAgewogICAgICAid2VpZ2h0IjogMC4wLAogICAgICAiaW1hZ2VfY29uZGl0aW9uIjogInJlYWwiLAogICAgICAiaGFyZG5lc3NfY29udHJvbCI6IG51bGwKICAgIH0sCiAgICB7CiAgICAgICJ3ZWlnaHQiOiAwLjUsCiAgICAgICJpbWFnZV9jb25kaXRpb24iOiAicmVhbCIsCiAgICAgICJoYXJkbmVzc19jb250cm9sIjogbnVsbAogICAgfSwKICAgIHsKICAgICAgIndlaWdodCI6IDAuMCwKICAgICAgImltYWdlX2NvbmRpdGlvbiI6ICJyZWFsIiwKICAgICAgImhhcmRuZXNzX2NvbnRyb2wiOiAibWVhbiIKICAgIH0KICBdLAogICJldmFsdWF0aW9uX3NwbGl0IjogInZhbGlkYXRpb24iLAogICJleGVjdXRpb25fc2NvcGUiOiB7CiAgICAicHVycG9zZSI6ICJzaW5nbGUtc2VlZCBjb3JyZWN0ZWQtdXNlciBkaWFnbm9zdGljLCBub3QgY29uZmlybWF0aW9uIiwKICAgICJzZWVkIjogMjAyNCwKICAgICJzZnRfZXBvY2hzIjogNCwKICAgICJkcG9fZXBvY2hzIjogMSwKICAgICJtYXhfcGl4ZWxzX2JlZm9yZV9wcm9jZXNzb3JfZ3JpZF9yb3VuZGluZyI6IDE2Mzg0LAogICAgInNmdF9jYXBfc2Vjb25kcyI6IDEwODAwLAogICAgInByb2JlX2NhcF9zZWNvbmRzIjogMTgwMCwKICAgICJmb2xsb3d1cF9jYXBfc2Vjb25kcyI6IDE2MjAwLAogICAgInRvdGFsX2FkZGl0aW9uYWxfY2FwX3NlY29uZHMiOiAyODgwMAogIH0KfQo=", "test_hanorec_fidelity.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwppbXBvcnQgdW5pdHRlc3QKZnJvbSB1bml0dGVzdC5tb2NrIGltcG9ydCBwYXRjaApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCkpCmltcG9ydCB0cmFpbiBhcyB0cmFpbl9tb2R1bGUgICMgbm9xYTogRTQwMgpmcm9tIHRyYWluIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIF9hY2N1bXVsYXRpb25fZ3JvdXBfc2l6ZSwKICAgIF9jdWRhX3BlYWtfbWVtb3J5LAogICAgX2J1aWxkX2Rwb19leGFtcGxlcywKICAgIF9jaHVua2VkLAogICAgX2Zpbml0ZSwKICAgIF9qc29uX2RlZmF1bHQsCiAgICBfcmVzZXRfY3VkYV9wZWFrX21lbW9yeSwKICAgIF9zY29yZV9yb3dzLAogICAgX3Jlc3RvcmVfYWN0aXZlX2FkYXB0ZXJzLAogICAgX3NldF9hY3RpdmVfYWRhcHRlcnMsCiAgICBfc2V0X29ubHlfdHJhaW5hYmxlX2FkYXB0ZXIsCiAgICBMT1JBX1RBUkdFVF9SRUdFWCwKICAgIF9idWlsZF9zZnRfZXhhbXBsZXMsCiAgICBfZGlzYWJsZV9kcm9wb3V0LAogICAgaGFyc19ub3JtYWxpemVfaGFyZG5lc3MsCiAgICByZXNwb25zaXZlbmVzcywKKQoKY2xhc3MgX0FkYXB0ZXJCYXNlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsKToKICAgICAgICBzZWxmLm1vZGVsID0gbW9kZWwKICAgICAgICBzZWxmLmNhbGxzID0gW10KCiAgICBkZWYgc2V0X2FkYXB0ZXIoc2VsZiwgYWRhcHRlcnMpOgogICAgICAgIG5hbWVzID0gW2FkYXB0ZXJzXSBpZiBpc2luc3RhbmNlKGFkYXB0ZXJzLCBzdHIpIGVsc2UgbGlzdChhZGFwdGVycykKICAgICAgICBzZWxmLmNhbGxzLmFwcGVuZChuYW1lcykKICAgICAgICBmb3IgbmFtZSwgcGFyYW1ldGVyIGluIHNlbGYubW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgICAgICBwYXJhbWV0ZXIucmVxdWlyZXNfZ3JhZCA9IGFueShmIi57YWRhcHRlcn0uIiBpbiBuYW1lIGZvciBhZGFwdGVyIGluIG5hbWVzKQoKY2xhc3MgX1BhcmFtZXRlcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLnJlcXVpcmVzX2dyYWQgPSBGYWxzZQoKCmNsYXNzIF9BZGFwdGVyTW9kZWw6CiAgICBkZWYgX19pbml0X18oc2VsZik6CiAgICAgICAgc2VsZi5jYWxscyA9IFtdCiAgICAgICAgc2VsZi5hY3RpdmVfYWRhcHRlciA9IE5vbmUKICAgICAgICBzZWxmLnBlZnRfY29uZmlnID0geyJzZnQiOiBvYmplY3QoKSwgImRwbyI6IG9iamVjdCgpfQogICAgICAgIHNlbGYucGFyYW1ldGVycyA9IHsKICAgICAgICAgICAgImxheWVyLnNmdC5sb3JhX0EiOiBfUGFyYW1ldGVyKCksCiAgICAgICAgICAgICJsYXllci5kcG8ubG9yYV9BIjogX1BhcmFtZXRlcigpLAogICAgICAgIH0KICAgICAgICBzZWxmLmJhc2VfbW9kZWwgPSBfQWRhcHRlckJhc2Uoc2VsZikKCiAgICBkZWYgc2V0X2FkYXB0ZXIoc2VsZiwgYWRhcHRlcik6CiAgICAgICAgc2VsZi5jYWxscy5hcHBlbmQoYWRhcHRlcikKCiAgICBkZWYgbmFtZWRfcGFyYW1ldGVycyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5wYXJhbWV0ZXJzLml0ZW1zKCkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBhY3RpdmVfcGVmdF9jb25maWcoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYucGVmdF9jb25maWdbc2VsZi5hY3RpdmVfYWRhcHRlcl0KCgpjbGFzcyBfQ3VkYVByb3BlcnRpZXM6CiAgICB0b3RhbF9tZW1vcnkgPSAxXzAwMAoKCmNsYXNzIF9GYWtlQ3VkYToKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLnJlc2V0ID0gRmFsc2UKCiAgICBkZWYgaXNfYXZhaWxhYmxlKHNlbGYpOgogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIHN5bmNocm9uaXplKHNlbGYpOgogICAgICAgIHJldHVybiBOb25lCgogICAgZGVmIHJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKHNlbGYpOgogICAgICAgIHNlbGYucmVzZXQgPSBUcnVlCgogICAgZGVmIGN1cnJlbnRfZGV2aWNlKHNlbGYpOgogICAgICAgIHJldHVybiAwCgogICAgZGVmIGdldF9kZXZpY2VfcHJvcGVydGllcyhzZWxmLCBfZGV2aWNlKToKICAgICAgICByZXR1cm4gX0N1ZGFQcm9wZXJ0aWVzKCkKCiAgICBkZWYgZ2V0X2RldmljZV9uYW1lKHNlbGYsIF9kZXZpY2UpOgogICAgICAgIHJldHVybiAiZmFrZS1ncHUiCgogICAgZGVmIG1heF9tZW1vcnlfYWxsb2NhdGVkKHNlbGYsIF9kZXZpY2UpOgogICAgICAgIHJldHVybiA2MDAKCiAgICBkZWYgbWF4X21lbW9yeV9yZXNlcnZlZChzZWxmLCBfZGV2aWNlKToKICAgICAgICByZXR1cm4gNzUwCgoKY2xhc3MgX0Zha2VUb3JjaDoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLmN1ZGEgPSBfRmFrZUN1ZGEoKQoKCgpjbGFzcyBfTm9HcmFkOgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKToKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGRlZiBfX2V4aXRfXyhzZWxmLCAqX2FyZ3MpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmNsYXNzIF9TY29yZVRvcmNoOgogICAgZGVmIG5vX2dyYWQoc2VsZik6CiAgICAgICAgcmV0dXJuIF9Ob0dyYWQoKQoKCmNsYXNzIF9TY29yZU1vZGVsOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYudHJhaW5pbmcgPSBUcnVlCgogICAgZGVmIGV2YWwoc2VsZik6CiAgICAgICAgc2VsZi50cmFpbmluZyA9IEZhbHNlCgogICAgZGVmIHRyYWluKHNlbGYpOgogICAgICAgIHNlbGYudHJhaW5pbmcgPSBUcnVlCgoKY2xhc3MgX1Njb3JlVmFsdWU6CiAgICBkZWYgX19pbml0X18oc2VsZiwgdmFsdWUpOgogICAgICAgIHNlbGYudmFsdWUgPSBmbG9hdCh2YWx1ZSkKCiAgICBkZWYgX19zdWJfXyhzZWxmLCBvdGhlcik6CiAgICAgICAgcmV0dXJuIF9TY29yZVZhbHVlKHNlbGYudmFsdWUgLSBvdGhlci52YWx1ZSkKCiAgICBkZWYgY3B1KHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLnZhbHVlCmNsYXNzIF9TY2FsYXI6CiAgICBkZWYgaXRlbShzZWxmKToKICAgICAgICByZXR1cm4gMS41CgoKY2xhc3MgSGFOb1JlY0ZpZGVsaXR5VGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfdXBzdHJlYW1fcmVzcG9uc2l2ZW5lc3NfaGFzX2dsb2JhbF90cmltbWVkX2JhdGNoX3NlbWFudGljcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHZhbHVlID0gcmVzcG9uc2l2ZW5lc3MoWzEuMCwgMi4wLCAzLjAsIDEwMC4wXSkKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIodmFsdWUsIDAuMCkKICAgICAgICBzZWxmLmFzc2VydExlc3ModmFsdWUsIDIuMCkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICByZXNwb25zaXZlbmVzcyhbMS4wLCBmbG9hdCgibmFuIiksIDMuMF0pCgogICAgZGVmIHRlc3RfZHBvX2V4YW1wbGVzX3ByZXNlcnZlX2JpbmFyeV9wYWlyX29yZGVyKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgZXhhbXBsZXMgPSBfYnVpbGRfZHBvX2V4YW1wbGVzKAogICAgICAgICAgICBbeyJoaXN0b3J5IjogWzEsIDIsIDNdLCAicG9zaXRpdmUiOiA0LCAibmVnYXRpdmUiOiA1fV0sCiAgICAgICAgICAgIHllc190b2tlbl9pZD0xMCwKICAgICAgICAgICAgbm9fdG9rZW5faWQ9MTEsCiAgICAgICAgICAgIHNodWZmbGVfbWFwPU5vbmUsCiAgICAgICAgKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoCiAgICAgICAgICAgIFsocm93WyJjYW5kaWRhdGUiXSwgcm93WyJjaG9zZW5fdG9rZW4iXSwgcm93WyJyZWplY3RlZF90b2tlbiJdKSBmb3Igcm93IGluIGV4YW1wbGVzXSwKICAgICAgICAgICAgWyg0LCAxMCwgMTEpLCAoNSwgMTEsIDEwKV0sCiAgICAgICAgKQoKICAgIGRlZiB0ZXN0X2JhdGNoX2JvdW5kYXJpZXNfYW5kX2hhcmRuZXNzX2FyZV9mYWlsX2Nsb3NlZChzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoW2xlbihiYXRjaCkgZm9yIGJhdGNoIGluIF9jaHVua2VkKGxpc3QocmFuZ2UoMTApKSwgNCwgVHJ1ZSldLCBbNCwgNF0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChbbGVuKGJhdGNoKSBmb3IgYmF0Y2ggaW4gX2NodW5rZWQobGlzdChyYW5nZSgxMCkpLCA0LCBGYWxzZSldLCBbNCwgNCwgMl0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oaGFyc19ub3JtYWxpemVfaGFyZG5lc3MoWzAuMCwgMS4wLCAyLjBdKSksIDMpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgX2NodW5rZWQoWzEsIDJdLCAwLCBUcnVlKQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIF9maW5pdGUoW10sICJ2YWx1ZXMiKQoKICAgIGRlZiB0ZXN0X211bHRpX2FkYXB0ZXJfYWN0aXZhdGlvbl9mcmVlemVzX3NmdF9wYXJhbWV0ZXJzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbW9kZWwgPSBfQWRhcHRlck1vZGVsKCkKICAgICAgICBfc2V0X2FjdGl2ZV9hZGFwdGVycyhtb2RlbCwgWyJzZnQiLCAiZHBvIl0pCiAgICAgICAgX3NldF9vbmx5X3RyYWluYWJsZV9hZGFwdGVyKG1vZGVsLCAiZHBvIikKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKG1vZGVsLnBhcmFtZXRlcnNbImxheWVyLnNmdC5sb3JhX0EiXS5yZXF1aXJlc19ncmFkKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShtb2RlbC5wYXJhbWV0ZXJzWyJsYXllci5kcG8ubG9yYV9BIl0ucmVxdWlyZXNfZ3JhZCkKICAgICAgICBfcmVzdG9yZV9hY3RpdmVfYWRhcHRlcnMobW9kZWwsIFsic2Z0IiwgImRwbyJdKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UobW9kZWwucGFyYW1ldGVyc1sibGF5ZXIuc2Z0LmxvcmFfQSJdLnJlcXVpcmVzX2dyYWQpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKG1vZGVsLnBhcmFtZXRlcnNbImxheWVyLmRwby5sb3JhX0EiXS5yZXF1aXJlc19ncmFkKQoKICAgIGRlZiB0ZXN0X211bHRpX2FkYXB0ZXJfYWN0aXZhdGlvbl91c2VzX3BlZnRfYmFzZV9tb2RlbChzZWxmKSAtPiBOb25lOgogICAgICAgIG1vZGVsID0gX0FkYXB0ZXJNb2RlbCgpCiAgICAgICAgX3NldF9hY3RpdmVfYWRhcHRlcnMobW9kZWwsIFsic2Z0IiwgImRwbyJdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobW9kZWwuYmFzZV9tb2RlbC5jYWxscywgW1sic2Z0IiwgImRwbyJdXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKG1vZGVsLmFjdGl2ZV9hZGFwdGVyLCAiZHBvIikKICAgICAgICBzZWxmLmFzc2VydElzKG1vZGVsLmFjdGl2ZV9wZWZ0X2NvbmZpZywgbW9kZWwucGVmdF9jb25maWdbImRwbyJdKQogICAgICAgIF9zZXRfYWN0aXZlX2FkYXB0ZXJzKG1vZGVsLCAic2Z0IikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKG1vZGVsLmNhbGxzLCBbInNmdCJdKQogICAgZGVmIHRlc3RfcGFydGlhbF9hY2N1bXVsYXRpb25fdXNlc19vbmVfZ3JvdXBfZGl2aXNvcihzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoCiAgICAgICAgICAgIFtfYWNjdW11bGF0aW9uX2dyb3VwX3NpemUocG9zaXRpb24sIDEwLCA0KSBmb3IgcG9zaXRpb24gaW4gcmFuZ2UoMTApXSwKICAgICAgICAgICAgWzQsIDQsIDQsIDQsIDQsIDQsIDQsIDQsIDIsIDJdLAogICAgICAgICkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICBfYWNjdW11bGF0aW9uX2dyb3VwX3NpemUoMCwgMCwgNCkKCgogICAgZGVmIHRlc3RfY3VkYV9wZWFrX21lbW9yeV9yZXBvcnRzX2NhcGFjaXR5X2ZyYWN0aW9ucyhzZWxmKSAtPiBOb25lOgogICAgICAgIHRvcmNoX21vZHVsZSA9IF9GYWtlVG9yY2goKQogICAgICAgIF9yZXNldF9jdWRhX3BlYWtfbWVtb3J5KHRvcmNoX21vZHVsZSkKICAgICAgICBzZWxmLmFzc2VydFRydWUodG9yY2hfbW9kdWxlLmN1ZGEucmVzZXQpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgKICAgICAgICAgICAgX2N1ZGFfcGVha19tZW1vcnkodG9yY2hfbW9kdWxlKSwKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImRldmljZSI6ICJmYWtlLWdwdSIsCiAgICAgICAgICAgICAgICAidG90YWxfYnl0ZXMiOiAxXzAwMCwKICAgICAgICAgICAgICAgICJwZWFrX2FsbG9jYXRlZF9ieXRlcyI6IDYwMCwKICAgICAgICAgICAgICAgICJwZWFrX3Jlc2VydmVkX2J5dGVzIjogNzUwLAogICAgICAgICAgICAgICAgInBlYWtfYWxsb2NhdGVkX2ZyYWN0aW9uIjogMC42LAogICAgICAgICAgICAgICAgInBlYWtfcmVzZXJ2ZWRfZnJhY3Rpb24iOiAwLjc1LAogICAgICAgICAgICB9LAogICAgICAgICkKCiAgICBkZWYgdGVzdF9jYW5kaWRhdGVfc2NvcmluZ19iYXRjaGVzX3dpdGhvdXRfY2hhbmdpbmdfcmFua19jb250cmFjdChzZWxmKSAtPiBOb25lOgogICAgICAgIG1vZGVsID0gX1Njb3JlTW9kZWwoKQogICAgICAgIGJhdGNoX3NpemVzID0gW10KCiAgICAgICAgZGVmIGZha2VfaW5wdXRzKF9wcm9jZXNzb3IsIF9tb2RlbCwgX2NhdGFsb2csIGV4YW1wbGVzLCBfcGl4ZWxzLCBfc2h1ZmZsZSk6CiAgICAgICAgICAgIGJhdGNoX3NpemVzLmFwcGVuZChsZW4oZXhhbXBsZXMpKQogICAgICAgICAgICByZXR1cm4gZXhhbXBsZXMKCiAgICAgICAgZGVmIGZha2VfbG9ncHJvYnMoX21vZGVsLCBleGFtcGxlcywgX2Z1bmN0aW9uYWwsIF95ZXMsIF9ubyk6CiAgICAgICAgICAgIHJldHVybiBbKF9TY29yZVZhbHVlKGV4YW1wbGVbImNhbmRpZGF0ZSJdKSwgX1Njb3JlVmFsdWUoMC4wKSkgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXNdCgogICAgICAgIHdpdGggKAogICAgICAgICAgICBwYXRjaC5vYmplY3QodHJhaW5fbW9kdWxlLCAiX2J1aWxkX2JhdGNoX2lucHV0cyIsIHNpZGVfZWZmZWN0PWZha2VfaW5wdXRzKSwKICAgICAgICAgICAgcGF0Y2gub2JqZWN0KHRyYWluX21vZHVsZSwgIl9iYXRjaF9hbnN3ZXJfbG9ncHJvYnMiLCBzaWRlX2VmZmVjdD1mYWtlX2xvZ3Byb2JzKSwKICAgICAgICApOgogICAgICAgICAgICByZXN1bHQgPSBfc2NvcmVfcm93cygKICAgICAgICAgICAgICAgIFt7CiAgICAgICAgICAgICAgICAgICAgInVzZXJfaWQiOiAidSIsCiAgICAgICAgICAgICAgICAgICAgImhpc3RvcnkiOiBbOSwgOCwgN10sCiAgICAgICAgICAgICAgICAgICAgInRhcmdldCI6IDIsCiAgICAgICAgICAgICAgICAgICAgImNhbmRpZGF0ZXMiOiBbMSwgMywgMl0sCiAgICAgICAgICAgICAgICB9XSwKICAgICAgICAgICAgICAgIG9iamVjdCgpLCBtb2RlbCwge30sIDgxOTIsIDEsIDAsIF9TY29yZVRvcmNoKCksIG9iamVjdCgpLCBOb25lLAogICAgICAgICAgICAgICAgZXZhbF9iYXRjaF9zaXplPTIsCiAgICAgICAgICAgICkKCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChiYXRjaF9zaXplcywgWzIsIDFdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WzBdWyJyYW5rZWRfY2FuZGlkYXRlcyJdLCBbMywgMiwgMV0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbMF1bInJhbmsiXSwgMikKICAgICAgICBzZWxmLmFzc2VydFRydWUobW9kZWwudHJhaW5pbmcpCgogICAgZGVmIHRlc3RfZXF1YWxfc2NvcmVzX2tlZXBfcmV0cmlldmVyX29yZGVyX25vdF9pdGVtX2lkX29yZGVyKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbW9kZWwgPSBfU2NvcmVNb2RlbCgpCiAgICAgICAgd2l0aCAoCiAgICAgICAgICAgIHBhdGNoLm9iamVjdCh0cmFpbl9tb2R1bGUsICJfYnVpbGRfYmF0Y2hfaW5wdXRzIiwgc2lkZV9lZmZlY3Q9bGFtYmRhIF9wLCBfbSwgX2MsIGV4LCBfeCwgX3M6IGV4KSwKICAgICAgICAgICAgcGF0Y2gub2JqZWN0KAogICAgICAgICAgICAgICAgdHJhaW5fbW9kdWxlLCAiX2JhdGNoX2Fuc3dlcl9sb2dwcm9icyIsCiAgICAgICAgICAgICAgICBzaWRlX2VmZmVjdD1sYW1iZGEgX20sIGV4LCBfZiwgX3ksIF9uOiBbKF9TY29yZVZhbHVlKDAuMCksIF9TY29yZVZhbHVlKDAuMCkpIGZvciBfIGluIGV4XSwKICAgICAgICAgICAgKSwKICAgICAgICApOgogICAgICAgICAgICByZXN1bHQgPSBfc2NvcmVfcm93cygKICAgICAgICAgICAgICAgIFt7InVzZXJfaWQiOiAidSIsICJoaXN0b3J5IjogWzFdLCAidGFyZ2V0IjogMywgImNhbmRpZGF0ZXMiOiBbOSwgMywgN119XSwKICAgICAgICAgICAgICAgIG9iamVjdCgpLCBtb2RlbCwge30sIDgxOTIsIDEsIDAsIF9TY29yZVRvcmNoKCksIG9iamVjdCgpLCBOb25lLAogICAgICAgICAgICApCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbMF1bInJhbmtlZF9jYW5kaWRhdGVzIl0sIFs5LCAzLCA3XSkKCiAgICBkZWYgdGVzdF9zZnRfdGFza19pc19iYWxhbmNlZF95ZXNfbm9fd2l0aF9zZWVkZWRfbmVnYXRpdmVzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcGFpcnMgPSBbCiAgICAgICAgICAgIHsiaGlzdG9yeSI6IFsxLCAyLCAzXSwgInBvc2l0aXZlIjogMTAgKyBpLCAic2Z0X25lZ2F0aXZlIjogMTAwICsgaSwgIm5lZ2F0aXZlIjogMjAwICsgaX0KICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UoNikKICAgICAgICBdCiAgICAgICAgZXhhbXBsZXMgPSBfYnVpbGRfc2Z0X2V4YW1wbGVzKHBhaXJzLCA3LCA4LCBzZWVkPTIwMjQpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChleGFtcGxlcywgX2J1aWxkX3NmdF9leGFtcGxlcyhwYWlycywgNywgOCwgc2VlZD0yMDI0KSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bShlWyJhbnN3ZXJfdG9rZW4iXSA9PSA3IGZvciBlIGluIGV4YW1wbGVzKSwgNikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bShlWyJhbnN3ZXJfdG9rZW4iXSA9PSA4IGZvciBlIGluIGV4YW1wbGVzKSwgNikKICAgICAgICBmb3IgZXhhbXBsZSBpbiBleGFtcGxlczoKICAgICAgICAgICAgZXhwZWN0ZWQgPSB7NzogezEwICsgaSBmb3IgaSBpbiByYW5nZSg2KX0sIDg6IHsxMDAgKyBpIGZvciBpIGluIHJhbmdlKDYpfX0KICAgICAgICAgICAgc2VsZi5hc3NlcnRJbihleGFtcGxlWyJjYW5kaWRhdGUiXSwgZXhwZWN0ZWRbZXhhbXBsZVsiYW5zd2VyX3Rva2VuIl1dKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoYW55KGVbImNhbmRpZGF0ZSJdID49IDIwMCBmb3IgZSBpbiBleGFtcGxlcyksICJoYXJkIG5lZ2F0aXZlcyBhcmUgRFBPLW9ubHkiKQoKICAgIGRlZiB0ZXN0X2Ryb3BvdXRfaXNfemVyb2VkX2luY2x1ZGluZ19sb3JhX2Ryb3BvdXQoc2VsZikgLT4gTm9uZToKICAgICAgICBjbGFzcyBEcm9wb3V0OgogICAgICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgcCk6CiAgICAgICAgICAgICAgICBzZWxmLnAgPSBwCgogICAgICAgIGNsYXNzIE1vZGVsOgogICAgICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kdWxlcyk6CiAgICAgICAgICAgICAgICBzZWxmLl9tb2R1bGVzID0gbW9kdWxlcwoKICAgICAgICAgICAgZGVmIG1vZHVsZXMoc2VsZik6CiAgICAgICAgICAgICAgICByZXR1cm4gaXRlcihzZWxmLl9tb2R1bGVzKQoKICAgICAgICBjbGFzcyBObjoKICAgICAgICAgICAgcGFzcwoKICAgICAgICBjbGFzcyBUb3JjaDoKICAgICAgICAgICAgbm4gPSBObigpCgogICAgICAgIFRvcmNoLm5uLkRyb3BvdXQgPSBEcm9wb3V0CiAgICAgICAgbW9kdWxlcyA9IFtEcm9wb3V0KDAuMDUpLCBEcm9wb3V0KDAuMCksIG9iamVjdCgpLCBEcm9wb3V0KDAuMSldCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChfZGlzYWJsZV9kcm9wb3V0KE1vZGVsKG1vZHVsZXMpLCBUb3JjaCksIDIpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFsbChtLnAgPT0gMC4wIGZvciBtIGluIG1vZHVsZXMgaWYgaXNpbnN0YW5jZShtLCBEcm9wb3V0KSkpCgogICAgZGVmIHRlc3RfbG9yYV9zY29wZV9pc19sYW5ndWFnZV9sYXllcnNfb25seShzZWxmKSAtPiBOb25lOgogICAgICAgIGltcG9ydCByZQoKICAgICAgICBsYW5ndWFnZSA9IFsKICAgICAgICAgICAgImJhc2VfbW9kZWwubW9kZWwubW9kZWwubGF5ZXJzLjAuc2VsZl9hdHRuLnFfcHJvaiIsCiAgICAgICAgICAgICJiYXNlX21vZGVsLm1vZGVsLm1vZGVsLmxheWVycy4zNS5tbHAuZG93bl9wcm9qIiwKICAgICAgICBdCiAgICAgICAgZnJvemVuID0gWwogICAgICAgICAgICAiYmFzZV9tb2RlbC5tb2RlbC52aXN1YWwuYmxvY2tzLjMubWxwLmdhdGVfcHJvaiIsCiAgICAgICAgICAgICJiYXNlX21vZGVsLm1vZGVsLnZpc3VhbC5ibG9ja3MuMy5hdHRuLnFrdiIsCiAgICAgICAgICAgICJiYXNlX21vZGVsLm1vZGVsLnZpc3VhbC5tZXJnZXIubWxwLjAiLAogICAgICAgICAgICAiYmFzZV9tb2RlbC5tb2RlbC5sbV9oZWFkIiwKICAgICAgICBdCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFsbChyZS5mdWxsbWF0Y2goTE9SQV9UQVJHRVRfUkVHRVgsIG5hbWUpIGZvciBuYW1lIGluIGxhbmd1YWdlKSkKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKGFueShyZS5mdWxsbWF0Y2goTE9SQV9UQVJHRVRfUkVHRVgsIG5hbWUpIGZvciBuYW1lIGluIGZyb3plbikpCgogICAgZGVmIHRlc3RfanNvbl9jb252ZXJ0ZXJfaGFuZGxlc19zY2FsYXJfcHJvdG9jb2xfYW5kX3JlamVjdHNfdW5rbm93bl92YWx1ZXMoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmFzc2VydEFsbW9zdEVxdWFsKF9qc29uX2RlZmF1bHQoX1NjYWxhcigpKSwgMS41KQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVHlwZUVycm9yKToKICAgICAgICAgICAgX2pzb25fZGVmYXVsdChvYmplY3QoKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==", "test_hanorec_protocol.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHVuaXR0ZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKZnJvbSBhdWRpdCBpbXBvcnQgYXVkaXRfYXJ0aWZhY3RzLCBwcm90b2NvbF9lcnJvcnMgICMgbm9xYTogRTQwMgpmcm9tIHByZXAgaW1wb3J0IF9idWlsZF9mcmVxdWVuY3lfbWF0Y2hlZF9kZXJhbmdlbWVudCwgX3NlZWRlZF9zZWxlY3Rpb24gICMgbm9xYTogRTQwMgoKUFJPVE9DT0wgPSBST09UIC8gInJ1bi1wcm90b2NvbC5qc29uIgoKCmNsYXNzIEhhTm9SZWNQcm90b2NvbFRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3Byb3RvY29sX2lzX2NvbXBsZXRlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcHJvdG9jb2wgPSBqc29uLmxvYWRzKFBST1RPQ09MLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByb3RvY29sX2Vycm9ycyhwcm90b2NvbCksIFtdKQoKICAgIGRlZiB0ZXN0X3NlbGVjdGlvbl9pc19zZWVkZWRfYW5kX25vdF9wcmVmaXhfYmlhc2VkKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm93cyA9IFsoaW5kZXgsIFtpbmRleCwgaW5kZXggKyAxLCBpbmRleCArIDIsIGluZGV4ICsgM10pIGZvciBpbmRleCBpbiByYW5nZSgyMCldCiAgICAgICAgZmlyc3QgPSBfc2VlZGVkX3NlbGVjdGlvbihyb3dzLCA1LCAyMDI0LCAidHJhaW4iKQogICAgICAgIHNlY29uZCA9IF9zZWVkZWRfc2VsZWN0aW9uKHJvd3MsIDUsIDIwMjQsICJ0cmFpbiIpCiAgICAgICAgb3RoZXIgPSBfc2VlZGVkX3NlbGVjdGlvbihyb3dzLCA1LCAyMDI1LCAidHJhaW4iKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZmlyc3QsIHNlY29uZCkKICAgICAgICBzZWxmLmFzc2VydE5vdEVxdWFsKGZpcnN0LCBvdGhlcikKICAgICAgICBzZWxmLmFzc2VydE5vdEVxdWFsKFtyb3dbMF0gZm9yIHJvdyBpbiBmaXJzdF0sIGxpc3QocmFuZ2UoNSkpKQoKICAgIGRlZiB0ZXN0X2ZyZXF1ZW5jeV9zaHVmZmxlX2lzX2JpamVjdGl2ZV9hbmRfZml4ZWRfcG9pbnRfZnJlZShzZWxmKSAtPiBOb25lOgogICAgICAgIGl0ZW1faWRzID0gbGlzdChyYW5nZSgxLCA5KSkKICAgICAgICBmcmVxdWVuY2llcyA9IHsxOiA1LCAyOiA1LCAzOiA0LCA0OiA0LCA1OiAzLCA2OiAzLCA3OiAyLCA4OiAyfQogICAgICAgIG1hcHBpbmcgPSBfYnVpbGRfZnJlcXVlbmN5X21hdGNoZWRfZGVyYW5nZW1lbnQoaXRlbV9pZHMsIGZyZXF1ZW5jaWVzLCAyMDI0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KG1hcHBpbmcpLCBzZXQoaXRlbV9pZHMpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KG1hcHBpbmcudmFsdWVzKCkpLCBzZXQoaXRlbV9pZHMpKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwoc291cmNlICE9IHRhcmdldCBmb3Igc291cmNlLCB0YXJnZXQgaW4gbWFwcGluZy5pdGVtcygpKSkKCiAgICBkZWYgdGVzdF9hcnRpZmFjdF9hdWRpdF9yZWplY3RzX3plcm9fYXJtcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHByb3RvY29sID0ganNvbi5sb2FkcyhQUk9UT0NPTC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0ZW1wb3Jhcnk6CiAgICAgICAgICAgIHJvb3QgPSBQYXRoKHRlbXBvcmFyeSkKICAgICAgICAgICAgKHJvb3QgLyAic2Z0X2J1bmRsZS5qc29uIikud3JpdGVfdGV4dCgie30iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICAocm9vdCAvICJzZnRfbG9yYV9zdGF0ZS5wdCIpLndyaXRlX2J5dGVzKGIic3RhdGUiKQogICAgICAgICAgICByZXN1bHQgPSBhdWRpdF9hcnRpZmFjdHMocm9vdCwgcHJvdG9jb2wpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WyJzdGF0dXMiXSwgIkZBSUwiKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoYW55KCJhcm0gbWF0cml4IG1pc21hdGNoIiBpbiBlcnJvciBmb3IgZXJyb3IgaW4gcmVzdWx0WyJlcnJvcnMiXSkpCgogICAgZGVmIHRlc3Rfc21va2VfYXVkaXRfcmVqZWN0c19taXNzaW5nX2NvbXB1dGVfdGVsZW1ldHJ5KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcHJvdG9jb2wgPSBqc29uLmxvYWRzKFBST1RPQ09MLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRlbXBvcmFyeToKICAgICAgICAgICAgcm9vdCA9IFBhdGgodGVtcG9yYXJ5KQogICAgICAgICAgICAocm9vdCAvICJzZnRfYnVuZGxlLmpzb24iKS53cml0ZV90ZXh0KCJ7fSIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIChyb290IC8gInNmdF9sb3JhX3N0YXRlLnB0Iikud3JpdGVfYnl0ZXMoYiJzdGF0ZSIpCiAgICAgICAgICAgIChyb290IC8gImFybS5wdCIpLndyaXRlX2J5dGVzKGIiY2hlY2twb2ludCIpCiAgICAgICAgICAgIChyb290IC8gImFybV9yZXN1bHQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAgICAgICAgICBqc29uLmR1bXBzKHsKICAgICAgICAgICAgICAgICAgICAid2VpZ2h0IjogMS4wLAogICAgICAgICAgICAgICAgICAgICJpbWFnZV9jb25kaXRpb24iOiAicmVhbCIsCiAgICAgICAgICAgICAgICAgICAgInN0YXR1cyI6ICJDT01QTEVURSIsCiAgICAgICAgICAgICAgICAgICAgInByZWRpY3Rpb25zIjogW3sKICAgICAgICAgICAgICAgICAgICAgICAgIm5kY2dAMTAiOiAwLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICJyZWNhbGxAMTAiOiAwLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICJjYW5kaWRhdGVfcmVjYWxsQDIwIjogMC4wLAogICAgICAgICAgICAgICAgICAgIH1dLAogICAgICAgICAgICAgICAgICAgICJjaGVja3BvaW50IjogImFybS5wdCIsCiAgICAgICAgICAgICAgICAgICAgInBhcmVudF9zZnRfc2hhMjU2IjogImhhc2giLAogICAgICAgICAgICAgICAgfSksCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIChyb290IC8gInNtb2tlX21hbmlmZXN0Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAganNvbi5kdW1wcyh7CiAgICAgICAgICAgICAgICAgICAgInB1cnBvc2UiOiAiY29ycmVjdG5lc3Nfb25seV9ub25fc2lnbmFsIiwKICAgICAgICAgICAgICAgICAgICAibXVzdF9ub3RfYmVfdXNlZF9hc19zaWduYWwiOiBUcnVlLAogICAgICAgICAgICAgICAgfSksCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlc3VsdCA9IGF1ZGl0X2FydGlmYWN0cyhyb290LCBwcm90b2NvbCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbInN0YXR1cyJdLCAiRkFJTCIpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oImludmFsaWQgc2Z0X2dwdV9tZW1vcnkgdGVsZW1ldHJ5IiwgcmVzdWx0WyJlcnJvcnMiXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRJbigiaW52YWxpZCBkcG9fc3RhZ2VfdGltaW5ncyBvcHRpbWl6ZXIgdGltaW5nIiwgcmVzdWx0WyJlcnJvcnMiXSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==", "test_hanorec_analysis.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQppbXBvcnQgc3lzCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCkpCmZyb20gYW5hbHlzaXMgaW1wb3J0IGZ1bGxfY29ob3J0X3ZhbHVlcywgaG9sbV9hZGp1c3QsIHBhaXJlZF9ib290c3RyYXAsIHJldHJpZXZlcl92ZXJkaWN0LCBzdW1tYXJpemUgICMgbm9xYTogRTQwMgoKCmRlZiBfcm93KHVzZXIsIHRhcmdldCwgbmRjZz0wLjAsIHJlY2FsbD0wLjAsIGNhbmQ9MS4wKToKICAgIHJldHVybiB7InVzZXJfaWQiOiB1c2VyLCAidGFyZ2V0IjogdGFyZ2V0LCAibmRjZ0AxMCI6IG5kY2csICJyZWNhbGxAMTAiOiByZWNhbGwsICJjYW5kaWRhdGVfcmVjYWxsQDIwIjogY2FuZH0KCgpkZWYgX3ZhbHMoZGVsdGFzKToKICAgIHJldHVybiBbeyJuZGNnQDEwIjogZCwgInJlY2FsbEAxMCI6IDAuMCwgImNhbmRpZGF0ZV9yZWNhbGxAMjAiOiAwLjB9IGZvciBkIGluIGRlbHRhc10KCgpjbGFzcyBBbmFseXNpc1Rlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3VuaW5mb3JtYXRpdmVfdXNlcnNfY291bnRfYXNfemVyb19pbl90aGVfZnVsbF9jb2hvcnQoc2VsZikgLT4gTm9uZToKICAgICAgICBjb2hvcnQgPSBbX3JvdygiYSIsIDEsIGNhbmQ9MC4wKSwgX3JvdygiYiIsIDIpLCBfcm93KCJjIiwgMywgY2FuZD0wLjApLCBfcm93KCJkIiwgNCwgY2FuZD0wLjApXQogICAgICAgIHZhbHVlcyA9IGZ1bGxfY29ob3J0X3ZhbHVlcyhjb2hvcnQsIFtfcm93KCJiIiwgMiwgbmRjZz0xLjAsIHJlY2FsbD0xLjApXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bW1hcml6ZSh2YWx1ZXMpWyJuZGNnQDEwIl0sIDAuMjUpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgZnVsbF9jb2hvcnRfdmFsdWVzKGNvaG9ydCwgW19yb3coInoiLCA5LCBuZGNnPTEuMCldKQoKICAgIGRlZiB0ZXN0X2Ryb3BwaW5nX2FuX2luZm9ybWF0aXZlX3VzZXJfaXNfYW5fZXJyb3Jfbm90X2Ffc2lsZW50X3plcm8oc2VsZikgLT4gTm9uZToKICAgICAgICBjb2hvcnQgPSBbX3JvdygiYSIsIDEpLCBfcm93KCJiIiwgMildCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgZnVsbF9jb2hvcnRfdmFsdWVzKGNvaG9ydCwgW19yb3coImIiLCAyLCBuZGNnPTEuMCldKQoKICAgIGRlZiB0ZXN0X2Jvb3RzdHJhcF9jaV9icmFja2V0c190aGVfdHJ1ZV9tZWFuX2FuZF9pc19kZXRlcm1pbmlzdGljKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm5nID0gcmFuZG9tLlJhbmRvbSg3KQogICAgICAgIGRlbHRhcyA9IFtybmcuY2hvaWNlKFstMC4yLCAwLjAsIDAuMCwgMC4zLCAwLjVdKSBmb3IgXyBpbiByYW5nZSgyMDApXQogICAgICAgIHRydWVfbWVhbiA9IHN1bShkZWx0YXMpIC8gbGVuKGRlbHRhcykKICAgICAgICByZXN1bHQgPSBwYWlyZWRfYm9vdHN0cmFwKF92YWxzKGRlbHRhcyksIF92YWxzKFswLjBdICogMjAwKSwgbj0yMDAwKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0LCBwYWlyZWRfYm9vdHN0cmFwKF92YWxzKGRlbHRhcyksIF92YWxzKFswLjBdICogMjAwKSwgbj0yMDAwKSkKICAgICAgICBsb3csIGhpZ2ggPSByZXN1bHRbImNpOTUiXQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyhsb3csIHRydWVfbWVhbikKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIoaGlnaCwgdHJ1ZV9tZWFuKQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyhoaWdoIC0gbG93LCAwLjE1KQogICAgICAgIHNlbGYuYXNzZXJ0QWxtb3N0RXF1YWwocmVzdWx0WyJtZWFuX2RlbHRhIl0sIHRydWVfbWVhbikKCiAgICBkZWYgdGVzdF9jb25zaXN0ZW50X2dhaW5faXNfZGV0ZWN0ZWRfYW5kX3BfdmFsdWVfaXNfbmV2ZXJfemVybyhzZWxmKSAtPiBOb25lOgogICAgICAgIHJlc3VsdCA9IHBhaXJlZF9ib290c3RyYXAoX3ZhbHMoWzAuNV0gKiA0MCksIF92YWxzKFswLjBdICogNDApLCBuPTUwMCkKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIocmVzdWx0WyJjaTk1Il1bMF0sIDApCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKHJlc3VsdFsicF90d29fc2lkZWQiXSwgMC4wKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmV0cmlldmVyX3ZlcmRpY3QocmVzdWx0KSwgIkJFQVRTX1JFVFJJRVZFUiIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbImNoYW5nZWRfZXZlbnRzIl0sIDQwKQoKICAgIGRlZiB0ZXN0X25vX2VmZmVjdF9jaV9jb250YWluc196ZXJvKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm5nID0gcmFuZG9tLlJhbmRvbSgzKQogICAgICAgIGRlbHRhcyA9IFtybmcuY2hvaWNlKFstMC4zLCAwLjNdKSBmb3IgXyBpbiByYW5nZSgzMDApXQogICAgICAgIHJlc3VsdCA9IHBhaXJlZF9ib290c3RyYXAoX3ZhbHMoZGVsdGFzKSwgX3ZhbHMoWzAuMF0gKiAzMDApLCBuPTIwMDApCiAgICAgICAgc2VsZi5hc3NlcnRMZXNzKHJlc3VsdFsiY2k5NSJdWzBdLCAwLjApCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKHJlc3VsdFsiY2k5NSJdWzFdLCAwLjApCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKHJlc3VsdFsicF90d29fc2lkZWQiXSwgMC4wNSkKCiAgICBkZWYgdGVzdF92ZXJkaWN0X2RvZXNfbm90X2hpZGVfZGVncmFkYXRpb25fbm9pc2Vfb3JfdHJpdmlhbF9nYWluKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXRyaWV2ZXJfdmVyZGljdCh7ImNpOTUiOiBbLTAuMDUsIC0wLjAxXX0pLCAiV09SU0VfVEhBTl9SRVRSSUVWRVIiKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmV0cmlldmVyX3ZlcmRpY3QoeyJjaTk1IjogWy0wLjA1LCAwLjAwMV19KSwgIk5PX01FQU5JTkdGVUxfR0FJTiIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXRyaWV2ZXJfdmVyZGljdCh7ImNpOTUiOiBbLTAuMDIsIDAuMDNdfSksICJJTkNPTkNMVVNJVkUiKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmV0cmlldmVyX3ZlcmRpY3QoeyJjaTk1IjogWzAuMDAwMSwgMC4wMDA1XX0pLCAiR0FJTl9CRUxPV19NRUFOSU5HRlVMIikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKAogICAgICAgICAgICByZXRyaWV2ZXJfdmVyZGljdCh7ImNpOTUiOiBbMC4wMDAxLCAwLjAwM119KSwKICAgICAgICAgICAgIlBPU0lUSVZFX0dBSU5fTUVBTklOR0ZVTE5FU1NfVU5DRVJUQUlOIiwKICAgICAgICApCgogICAgZGVmIHRlc3RfbWVhbmluZ2Z1bF9jaV9jYW5ub3Rfb3ZlcnJpZGVfdW5yZWplY3RlZF9udWxsKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgY29udHJhc3QgPSB7ImNpOTUiOiBbMC4wMDMsIDAuMDJdLCAicF90d29fc2lkZWQiOiAwLjA2MjV9CiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXRyaWV2ZXJfdmVyZGljdChjb250cmFzdCksICJQT1NJVElWRV9DSV9OVUxMX05PVF9SRUpFQ1RFRCIpCiAgICAgICAgY29udHJhc3RbInBfdHdvX3NpZGVkIl0gPSAwLjAyCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXRyaWV2ZXJfdmVyZGljdChjb250cmFzdCksICJCRUFUU19SRVRSSUVWRVIiKQogICAgICAgIGNvbnRyYXN0WyJjaTk1Il0gPSBbMC4wMDAxLCAwLjAyXQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmV0cmlldmVyX3ZlcmRpY3QoY29udHJhc3QpLCAiUE9TSVRJVkVfR0FJTl9NRUFOSU5HRlVMTkVTU19VTkNFUlRBSU4iKQoKICAgIGRlZiB0ZXN0X2R1cGxpY2F0ZV9ldmVudHNfYXJlX3JlamVjdGVkKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm93ID0gX3JvdygiYSIsIDEsIG5kY2c9MC4yKQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIGZ1bGxfY29ob3J0X3ZhbHVlcyhbcm93XSwgW3Jvdywgcm93XSkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICBmdWxsX2NvaG9ydF92YWx1ZXMoW3Jvdywgcm93XSwgW3Jvd10pCgogICAgZGVmIHRlc3RfcmVwZWF0ZWRfZXZlbnRzX2RvX25vdF9jcmVhdGVfaW5kZXBlbmRlbnRfdXNlcnMoc2VsZikgLT4gTm9uZToKICAgICAgICBiZXR0ZXIgPSBbeyoqX3ZhbHMoWzAuM10pWzBdLCAiY2x1c3Rlcl9pZCI6IHN0cihpIC8vIDEwKX0gZm9yIGkgaW4gcmFuZ2UoMjApXQogICAgICAgIGJhc2UgPSBbeyoqcm93LCAibmRjZ0AxMCI6IDAuMH0gZm9yIHJvdyBpbiBiZXR0ZXJdCiAgICAgICAgcmVzdWx0ID0gcGFpcmVkX2Jvb3RzdHJhcChiZXR0ZXIsIGJhc2UsIG49MTAwMCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdFsiaW5kZXBlbmRlbnRfY2x1c3RlcnMiXSwgMikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdFsiZXZlbnRzIl0sIDIwKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WyJyZXNhbXBsaW5nX3VuaXQiXSwgInNvdXJjZV91c2VyIikKICAgICAgICAjIEZvdXIgZXF1YWxseSBsaWtlbHkgbnVsbCBsYWJlbCBhc3NpZ25tZW50czsgYm90aCBlcXVhbC1zaWduIGFzc2lnbm1lbnRzIGFyZSBleHRyZW1lLgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WyJwX3R3b19zaWRlZCJdLCAwLjUpCgogICAgZGVmIHRlc3RfcGFydGlhbF9vcl9taXNtYXRjaGVkX2NsdXN0ZXJfbWV0YWRhdGFfZmFpbHMoc2VsZikgLT4gTm9uZToKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICBwYWlyZWRfYm9vdHN0cmFwKFt7Im5kY2dAMTAiOiAwLjMsICJjbHVzdGVyX2lkIjogImEifV0sIFt7Im5kY2dAMTAiOiAwLjB9XSkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICBwYWlyZWRfYm9vdHN0cmFwKAogICAgICAgICAgICAgICAgW3sibmRjZ0AxMCI6IDAuMywgImNsdXN0ZXJfaWQiOiAiYSJ9XSwKICAgICAgICAgICAgICAgIFt7Im5kY2dAMTAiOiAwLjAsICJjbHVzdGVyX2lkIjogImIifV0sCiAgICAgICAgICAgICkKCiAgICBkZWYgdGVzdF9ob2xtX2lzX21vbm90b25lX2FuZF9jYXBwZWQoc2VsZikgLT4gTm9uZToKICAgICAgICBhZGp1c3RlZCA9IGhvbG1fYWRqdXN0KHsieCI6IDAuMDEsICJ5IjogMC4wNH0pCiAgICAgICAgc2VsZi5hc3NlcnRBbG1vc3RFcXVhbChhZGp1c3RlZFsieCJdLCAwLjAyKQogICAgICAgIHNlbGYuYXNzZXJ0QWxtb3N0RXF1YWwoYWRqdXN0ZWRbInkiXSwgMC4wNCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGhvbG1fYWRqdXN0KHsieCI6IDAuOSwgInkiOiAwLjk1fSksIHsieCI6IDEuMCwgInkiOiAxLjB9KQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB1bml0dGVzdC5tYWluKCkK", "test_hanorec_identity.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgY3N2CmltcG9ydCBoYXNobGliCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCkpCmZyb20gcHJlcCBpbXBvcnQgX2lkZW50aXR5X3Jvd3MsIF91c2VyX3NlZW5faXRlbXMKCgpjbGFzcyBJZGVudGl0eVRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3NsaWRpbmdfd2luZG93c19hbmRfZnV0dXJlX3RyYWluX3Bvc2l0aXZlc19hcmVfc2VlbihzZWxmKToKICAgICAgICByb3dzID0gW1sxLCAyLCAzLCA0XSwgWzMsIDQsIDUsIDZdLCBbMSwgMiwgMywgN11dCiAgICAgICAgc2VlbiA9IF91c2VyX3NlZW5faXRlbXMocm93cywgWyJzYW1lLXVzZXIiLCAic2FtZS11c2VyIiwgImFub3RoZXItdXNlciJdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2Vlblsic2FtZS11c2VyIl0sIHsxLCAyLCAzLCA0LCA1LCA2fSkKICAgICAgICBzZWxmLmFzc2VydE5vdEluKDcsIHNlZW5bInNhbWUtdXNlciJdKQogICAgICAgIHNlbGYuYXNzZXJ0Tm90SW4oNiwgc2VlblsiYW5vdGhlci11c2VyIl0pCgogICAgZGVmIHRlc3RfY3N2X2FsaWdubWVudF9vZmZzZXRfYW5kX3JlYWxfaWRlbnRpdHkoc2VsZik6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0ZW1wOgogICAgICAgICAgICByb290ID0gUGF0aCh0ZW1wKQogICAgICAgICAgICBmb2xkZXIgPSByb290IC8gInRyYWluIgogICAgICAgICAgICBmb2xkZXIubWtkaXIoKQogICAgICAgICAgICBwYXRoID0gZm9sZGVyIC8gIlZpZGVvX0dhbWVzXzVfMTk5Ni05LTIwMjMtMTAuY3N2IgogICAgICAgICAgICB3aXRoIHBhdGgub3BlbigidyIsIG5ld2xpbmU9IiIpIGFzIGhhbmRsZToKICAgICAgICAgICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKGhhbmRsZSwgZmllbGRuYW1lcz1bInVzZXJfaWQiLCAiaGlzdG9yeV9pdGVtX2lkIiwgIml0ZW1faWQiXSkKICAgICAgICAgICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgICAgICAgICB3cml0ZXIud3JpdGVyb3coeyJ1c2VyX2lkIjogImFsaWNlIiwgImhpc3RvcnlfaXRlbV9pZCI6ICJbMCwgMV0iLCAiaXRlbV9pZCI6IDJ9KQogICAgICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KHsidXNlcl9pZCI6ICJhbGljZSIsICJoaXN0b3J5X2l0ZW1faWQiOiAiWzEsIDJdIiwgIml0ZW1faWQiOiAzfSkKICAgICAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdyh7InVzZXJfaWQiOiAiYm9iIiwgImhpc3RvcnlfaXRlbV9pZCI6ICJbMCwgMV0iLCAiaXRlbV9pZCI6IDR9KQogICAgICAgICAgICBwaW4gPSBoYXNobGliLnNoYTI1NihwYXRoLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKICAgICAgICAgICAgY2x1c3RlcnMsIF8gPSBfaWRlbnRpdHlfcm93cyhyb290LCAidHJhaW4iLCBbWzEsIDIsIDNdLCBbMiwgMywgNF0sIFsxLCAyLCA1XV0sIHBpbikKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChjbHVzdGVyc1swXSwgY2x1c3RlcnNbMV0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0Tm90RXF1YWwoY2x1c3RlcnNbMF0sIGNsdXN0ZXJzWzJdKQogICAgICAgICAgICBzZWxmLmFzc2VydE5vdEluKCJhbGljZSIsIGNsdXN0ZXJzKQogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFJ1bnRpbWVFcnJvcik6CiAgICAgICAgICAgICAgICBfaWRlbnRpdHlfcm93cyhyb290LCAidHJhaW4iLCBbWzEsIDIsIDRdLCBbMiwgMywgNF0sIFsxLCAyLCA1XV0sIHBpbikKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhSdW50aW1lRXJyb3IpOgogICAgICAgICAgICAgICAgX2lkZW50aXR5X3Jvd3Mocm9vdCwgInRyYWluIiwgW1sxLCAyLCAzXV0sIHBpbikKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhSdW50aW1lRXJyb3IpOgogICAgICAgICAgICAgICAgX2lkZW50aXR5X3Jvd3Mocm9vdCwgInRyYWluIiwgW1sxLCAyLCAzXSwgWzIsIDMsIDRdLCBbMSwgMiwgNV1dLCAiYmFkLWhhc2giKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB1bml0dGVzdC5tYWluKCkK"}
SOURCE_SHA256 = {
 "prep.py": "5c5b6f51a03b9f408f12d12664128d77dcbfa5917407675f5c8d6030b621a207",
 "train.py": "eb1769a7566fec2d8ce8769eb9610cb94468f024f404057d43a91cc85a739bf6",
 "analysis.py": "2f62c6199df405ac7cbb42a3f1d59449ffbdc4f1284eda44621ecc8d3c3ad342",
 "experiment.json": "4288414f4b57a9c6074454ced2773ace80193346045fa2ae3ba0a7a698b984ec",
 "audit.py": "a3f0aa6c4de5a0a8dc970c050921eb8784dabb99cb197992d5dd15678f1e87aa",
 "run-protocol.json": "97832d52d8cb0f66b379d9364baf03c8f435e09835f46e9c4aa9c3130c6efd71",
 "test_hanorec_fidelity.py": "fe2e99b8c37752792074b29561dec3bb8be8654357cb2308ca901f438a08ddc3",
 "test_hanorec_protocol.py": "f7ba56e9091420c26dea5181672d2f3e470ddd771d8c88f35daf4617ae76cd29",
 "test_hanorec_analysis.py": "ce99e8ebe660645ae0168af8375e883a1c0c94a0cb068a0906f36807d3e11fd9",
 "test_hanorec_identity.py": "9a784ed6d15a757e2907897d220d28ba8476379c49886e10d53d3181b48033ce"
}
import base64
SOURCE_DIR = Path("/kaggle/working/v11_source")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for name, blob in EMBEDDED_SOURCES.items():
    data = base64.b64decode(blob)
    assert hashlib.sha256(data).hexdigest() == SOURCE_SHA256[name], f"embedded source corrupted: {name}"
    (SOURCE_DIR / name).write_bytes(data)
sys.path.insert(0, str(SOURCE_DIR))
print({name: SOURCE_SHA256[name][:12] for name in SOURCE_SHA256})

## 3. CPU regression tests

In [ ]:
# CPU regression tests for every repaired behavior (SFT task, dropout, LoRA scope, tie order, statistics).
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(SOURCE_DIR), "-p", "test_hanorec_*.py"],
               cwd=SOURCE_DIR, check=True)

## 4. Protocol configuration

In [ ]:
import copy
BASE = json.loads((SOURCE_DIR / "experiment.json").read_text())
CONFIG = copy.deepcopy(BASE)
CONFIG.update({
    "seed": 2024,
    "train_pairs": 530, "validation_users": 265,
    "eval_users": 1,                      # test rows are built by prep but never scored (validation-only decisions)
    "history_items": 3, "smoke": False,
    "sft_epochs": 4, "dpo_epochs": 1,  # matched historical schedule; corrections are data/statistics only
    "sft_batch_size": 2, "sft_gradient_accumulation_steps": 8,   # train/mllm_sft.sh
    "batch_size": 4, "gradient_accumulation_steps": 8,           # configs/hanorec/*_hit1.yaml
    "eval_batch_size": 1, "verify_scoring_batching": False,
    "budget_seconds": 10800,
    "learning_probe_pairs": 8,
    "weights": [1.0, 0.5, 0.0],
    "sft_probe_pairs": 32,
})
print(json.dumps({k: CONFIG[k] for k in ("seed", "train_pairs", "validation_users", "sft_epochs", "dpo_epochs",
                                         "sft_batch_size", "batch_size", "gradient_accumulation_steps", "learning_rate", "noise_sigma", "beta0")}, indent=1))

## 5. Prepare data (train-only pairs, seeded SFT negatives, validation candidates)
Pinned artifacts are verified by SHA-256 inside `prep.prepare`. Test rows are read by `prep` to build its
inference-asset object but are **removed before any model call**. Users whose target is not in SASRec's
top-20 cannot be changed by a reranker, so Qwen scores only the informative users; metrics are expanded to
the full validation cohort (absent users contribute 0 to every arm *and* to SASRec).

In [ ]:
from prep import prepare
import train as T
import analysis as A

OUT_ROOT = Path("/kaggle/working/v11"); OUT_ROOT.mkdir(parents=True, exist_ok=True)
# Reuse ONLY image bytes from the archived SFT output, never its trained adapter or pairs.
import shutil
archived_manifests = sorted(Path("/kaggle/input").rglob("job_manifest.json"))
assert len(archived_manifests) == 1, "expected one archived image-cache parent"
archived_path = archived_manifests[0]
assert hashlib.sha256(archived_path.read_bytes()).hexdigest() == "7ec0525def7ff2fab2ba1fed7bfa5f1b43bb13ceabada136c9a567f9b7a06fba"
archived_bundle = json.loads((archived_path.parent / "sft_real/sft_bundle.json").read_text())
assert hashlib.sha256((archived_path.parent / "sft_real/sft_bundle.json").read_bytes()).hexdigest() == json.loads(archived_path.read_text())["sft_bundle_sha256"]
reused_files = set()
for item, record in archived_bundle["catalog"].items():
    source = archived_path.parent / "sft_real" / record["image_path"]
    if not source.is_file():
        continue  # prepare downloads missing bytes; hashes are checked again below
    assert hashlib.sha256(source.read_bytes()).hexdigest() == archived_bundle["provenance"]["image_provenance"][item]["sha256"]
    target = OUT_ROOT / "images" / source.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        shutil.copyfile(source, target)
    reused_files.add(source.name)
t0 = time.monotonic()
prepared = prepare(CONFIG, Path("/kaggle/input"), OUT_ROOT, SOURCE_DIR)
archived_hashes = archived_bundle["provenance"]["image_provenance"]
for item, image in prepared["provenance"]["image_provenance"].items():
    if item in archived_hashes:
        assert image["sha256"] == archived_hashes[item]["sha256"], f"image bytes drifted for item {item}"
ALL_VALIDATION = list(prepared["validation"])
SASREC_VALIDATION = list(prepared["baselines"]["sasrec_validation"])
INFORMATIVE = [r for r in ALL_VALIDATION if int(r["target"]) in set(map(int, r["candidates"]))]
prepared["validation"] = INFORMATIVE
prepared["evaluation"] = []
prepared["baselines"]["sasrec_test"] = []
sft_pool_clean = all(p["sft_negative"] not in set(p["excluded_items"]) for p in prepared["train"])
PREP_REPORT = {
    "prepare_seconds": round(time.monotonic() - t0, 1),
    "reused_unique_images": len(reused_files),
    "train_pairs": len(prepared["train"]), "catalog_items": len(prepared["catalog"]),
    "validation_events": len(ALL_VALIDATION), "true_validation_users": len({r["cluster_id"] for r in ALL_VALIDATION}),
    "informative_events": len(INFORMATIVE),
    "candidate_recall@20": len(INFORMATIVE) / len(ALL_VALIDATION),
    "sft_negatives_outside_excluded_items": bool(sft_pool_clean),
    "sft_negative_equals_hard_negative": sum(p["sft_negative"] == p["negative"] for p in prepared["train"]),
    "test_model_scoring": "disabled",
}
print(json.dumps(PREP_REPORT, indent=1))
assert sft_pool_clean and len(INFORMATIVE) >= 20, "too few informative users for a usable gate"

## 6. Fixed image budget
Keep `max_pixels=16384`, the previously measured T4 setting. No resolution search in this diagnostic.
Actual peak memory is recorded again; historical DPO used up to 96.8% reserved VRAM.

In [ ]:
import gc
# Preserve the already measured resolution instead of searching for a favorable new recipe.
CONFIG["max_pixels"] = 16384
PIXEL_ADAPTATION = {"frozen": 262144, "used": 16384, "reason": "fixed T4 recipe from audited prior measurement"}
gc.collect(); torch.cuda.empty_cache()
print({"max_pixels": CONFIG["max_pixels"], "adaptation": PIXEL_ADAPTATION})

## 6b. Preflight every catalog image before the fixed four-epoch diagnostic
Every image (train and validation-only) is processed on CPU before spending training time.
The recipe is fixed at four SFT epochs to isolate corrected labels/user identity from schedule changes.

In [ ]:
from transformers import AutoProcessor
preflight_processor = AutoProcessor.from_pretrained(CONFIG["model_id"], revision=CONFIG["model_revision"])
bad = []
for item_id, record in prepared["catalog"].items():
    try:
        preflight_processor.image_processor(images=[T._resize_image(record["image_path"], CONFIG["max_pixels"])], return_tensors="pt")
    except Exception as exc:  # noqa: BLE001 - report every unusable image at once
        bad.append((item_id, repr(exc)[:160]))
assert not bad, f"{len(bad)} catalog images cannot be processed: {bad[:5]}"
del preflight_processor
T_EX = 1.7547347672499995  # historical measured fwd+bwd s/example; actual timing recorded again
ADAPTATIONS = {
    "gpu_bf16": None if GPU["bf16_native"] else "bf16 emulated on " + GPU["name"] + " (Kaggle API offers only T4/P100)",
    "max_pixels": PIXEL_ADAPTATION,
    "sft_epochs": {"frozen": 5, "used": CONFIG["sft_epochs"], "reason": "fixed matched four-epoch schedule; three-hour cap"},
}
print({"catalog_images_ok": len(prepared["catalog"]), "seconds_per_example": round(T_EX, 3),
       "sft_epochs": CONFIG["sft_epochs"], "adaptations": ADAPTATIONS})

## 7. SFT parent (real images)
Task = HaNoRec hit=1: true next item -> `Yes`, seeded random unseen item -> `No` (50/50), four fixed epochs,
micro-batch 2 x accumulation 8, cosine LR, LoRA r=8/alpha=32/dropout 0.05 on the **language-model** layers only.

In [ ]:
SFT_DIR = OUT_ROOT / "sft_real"
CONFIG["budget_seconds"] = max(1, 10800 - (time.monotonic() - NOTEBOOK_T0) - 1200)
sft = T.run_sft({**CONFIG, "sft_image_condition": "real"}, prepared, SFT_DIR, SOURCE_DIR)
assert sft["status"] == "COMPLETE" and len(sft["sft_validation_predictions"]) == len(INFORMATIVE)
print({"updates": sft["sft_optimizer_updates"], "lora_modules": sft["lora_module_count"],
       "reference_probe": sft["reference_reproducibility_check"], "elapsed_min": round((time.monotonic() - NOTEBOOK_T0) / 60, 1)})
print(json.dumps(sft["sft_probe"], indent=1))

## 8. Gate: is the SFT reference still saturated, and where does SFT sit against SASRec?
`positive_yes_fraction` / `sft_negative_no_fraction` are measured on 32 *training* pairs: a saturation detector
(the old reference answered `Yes` to everything), not a generalisation test. Generalisation is the SFT-only
vs SASRec paired bootstrap on the full validation cohort. Nothing is tuned from either.

In [ ]:
probe_stats = sft["sft_probe"]
GATE = {
    "positive_yes_fraction": probe_stats["positive_yes_fraction"],
    "sft_negative_no_fraction": probe_stats["sft_negative_no_fraction"],
    "hard_negative_no_fraction": probe_stats["hard_negative_no_fraction"],
}
GATE["reference_learned_binary_task"] = bool(
    GATE["positive_yes_fraction"] >= 0.6 and GATE["sft_negative_no_fraction"] >= 0.6)
sasrec_vals = A.full_cohort_values(SASREC_VALIDATION, SASREC_VALIDATION)
sft_vals = A.full_cohort_values(SASREC_VALIDATION, sft["sft_validation_predictions"])
sft_vs_sasrec = {m: A.paired_bootstrap(sft_vals, sasrec_vals, m) for m in A.METRICS}
SUMMARY = {
    "gate": GATE,
    "sasrec_order": A.summarize(sasrec_vals), "sft_only": A.summarize(sft_vals),
    "sft_minus_sasrec": sft_vs_sasrec,
    "sft_vs_sasrec_verdict": A.retriever_verdict(sft_vs_sasrec["ndcg@10"]),
    "informative_events": len(INFORMATIVE), "validation_events": len(ALL_VALIDATION),
    "true_validation_users": len({row["cluster_id"] for row in ALL_VALIDATION}),
}
print(json.dumps(SUMMARY, indent=1))

## 9. Package artifacts
`job_manifest.json` pins source hashes, GPU, chosen pixels and timings. The arm job mounts this kernel's
output (SFT bundle, adapter, images) and refuses to start if hashes disagree.

In [ ]:
MANIFEST = {
    "job": "sft", "status": "COMPLETE", "sft_gate_passed": GATE["reference_learned_binary_task"],
    "must_not_be_used_as_confirmation": True,
    "stage_gpu_cap_seconds": 10800, "total_additional_gpu_cap_seconds": 28800,
    "source_sha256": SOURCE_SHA256, "gpu": GPU,
    "config": {k: v for k, v in CONFIG.items() if k != "artifact_pins"},
    "adaptations": ADAPTATIONS, "seconds_per_example": T_EX,
    "prep_report": PREP_REPORT, "summary": SUMMARY,
    "sft_bundle_sha256": hashlib.sha256((SFT_DIR / "sft_bundle.json").read_bytes()).hexdigest(),
    "sft_state_sha256": sft["sft_state_sha256"],
    "elapsed_minutes": round((time.monotonic() - NOTEBOOK_T0) / 60, 1),
}
(OUT_ROOT / "job_manifest.json").write_text(json.dumps(MANIFEST, indent=2, default=str))
print("PACKAGED", MANIFEST["elapsed_minutes"], "min; SFT gate:", GATE["reference_learned_binary_task"],
      "| SFT vs SASRec:", SUMMARY["sft_vs_sasrec_verdict"])